<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-09/NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html)

# Week 09: Model reliability and explainable artificial intelligence (XAI)

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lab.html) · [PDF notes](Week09_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-09/NB09_reliability_and_explainable_ai.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

**Together with the lecture page.** You come to this Jupyter notebook from the practice stops of the lecture page. At the end of every stop, a link takes you back to the lecture page, to the place where reading continues.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. The exact Shapley values of Section 3 take about half a minute, and the echo network of Section 5 trains in under a minute on an ordinary laptop; no graphics card is needed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages. A link of the lecture page opens the notebook at a stop or a section: There, run the first cell, the preparation cell or the catch-up cell. It also runs Section 0 if this has not happened in the session.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. The second cell also downloads the 300 patients of CAMUS of Week 4 again, about 250 MB, in one to three minutes; files that are already there are skipped. The Cleveland data and the chest pain unit come with the course package.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import itertools
import math

from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import camus, triage
from vclinic.datasets import load_cleveland

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch is not installed. Install it as SETUP.md describes, then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 9, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
camus.download()
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["shap"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 9.1": (56, 0),
    "Exercise 9.2": (0.105, 0.001),
    "Exercise 9.3": (0.00357, 1e-05),
    "Exercise A": (33, 0),
    "Exercise B": (0.749, 0.002),
    "Exercise C": (56, 3),
    "Exercise D": (0.031, 0.02),
    "Exercise E": (0.824, 0.06),
    "Exercise F": (0.67, 0.08),
    "Exercise G": (2, 0),
    "Exercise H": (0.687, 0.01),
    "Exercise I": (723, 0),
    "Exercise J": (0.932, 0.02),
    "Step 1.1": (0.904, 0.03),
    "Step 1.2": (0.155, 0.03),
    "Step 1.3": (-1.175, 0.01),
    "Step 1.4": (0.138, 0.03),
    "Step 2.1": (0.145, 0.03),
    "Step 2.2": (0.7, 0.25),
    "Step 2.3": (0.796, 0.06),
    "Step 2.4": (0.317, 0.06),
    "Step 3.1": (0.988, 0.02),
    "Step 3.2": (253, 0),
    "Step 3.3": (0.956, 0.001),
    "Step 3.4": (0.417, 0.02),
    "Step 1.5": (9, 0),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the data of the week."""
    g = globals()
    k = key
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Run the sections before this exercise first."))
    sets = {"training": "training", "validation": "validation", "testing": "test", "test": "test"}
    if k == "9.2":
        rows = [(0, 0, 0, 0.552), (1, 0, 0, 0.660), (0, 1, 0, 0.617), (0, 0, 1, 0.456), (1, 1, 0, 0.799), (1, 0, 1, 0.584), (0, 1, 1, 0.533), (1, 1, 1, 0.727)]
        return pd.DataFrame(rows, columns=["chest pain type", "exercise angina", "sex", "risk"])
    if k == "A":
        forest, X_test, y_test = need("forest", "X_test", "y_test")
        return pd.DataFrame({"patient": X_test.index, "probability": forest.predict_proba(X_test)[:, 1].round(3),
                             "disease": y_test.to_numpy()})
    if k in ("B", "C", "D"):
        heart, X_train, X_test = need("heart", "X_train", "X_test")
        h = heart.loc[list(X_train.index) + list(X_test.index)]
        cp = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}
        return pd.DataFrame({"patient": h.index, "set": np.where(h.index.isin(X_train.index), "training", "test"),
                             "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                             "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                             "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                             "ST depression": h["oldpeak"], "chest pain type": h["cp"].map(cp),
                             "disease": h["disease"]}).reset_index(drop=True)
    if k in ("E", "F", "G"):
        views, echo_x, outline, echo_y, echo_set = need("views", "echo_x", "outline", "echo_y", "echo_set")
        q = camus.table().set_index("patient")
        return pd.DataFrame({"patient": [p for p, v in views], "view": [v for p, v in views], "set": [sets[s] for s in echo_set],
                             "image quality": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in views],
                             "end-diastole": list(echo_x[:, 0]), "end-systole": list(echo_x[:, 1]),
                             "outline": list(outline.astype(np.uint8)), "low ejection fraction": echo_y})
    if k == "H":
        A, B = need("A", "B")
        rows = []
        for name, d in (("A", A), ("B", B)):
            url = d["troponin_url"]
            rows.append(pd.DataFrame({"hospital": name, "set": d["set"].map(sets).to_numpy() if name == "A" else "hospital B",
                                      "age": d["age"], "sex": d["sex"].map({"M": "man", "F": "woman"}),
                                      "smoker": d["smoker"], "diabetes": d["diabetes"], "hypertension": d["hypertension"],
                                      "hyperlipidaemia": d["hyperlipidaemia"], "obesity": d["obesity"],
                                      "family history": d["family_history"], "known coronary artery disease": d["known_cad"],
                                      "heart rate": d["heart_rate"], "systolic pressure": d["sbp"], "eGFR": d["egfr"],
                                      "troponin at arrival": (d["troponin_0h"] / url).round(2),
                                      "troponin after one hour": (d["troponin_1h"] / url).round(2),
                                      "echocardiogram done": d["echo_done"], "ejection fraction": d["lvef"],
                                      "wall motion abnormality": d["rwma"], "triage note": d["note_tr"]}))
        return pd.concat(rows, ignore_index=True)
    if k == "I":
        B, pB = need("B", "pB")
        return pd.DataFrame({"order": np.arange(1, len(B) + 1), "probability": pB, "MACE": B["mace30"].to_numpy()})
    if k == "J":
        A, pA, va, te = need("A", "pA", "va", "te")
        m = va | te
        return pd.DataFrame({"set": np.where(va, "validation", "test")[m], "sex": A["sex"].map({"M": "man", "F": "woman"}).to_numpy()[m],
                             "age": A["age"].to_numpy()[m], "probability": pA[m], "MACE": A["mace30"].to_numpy()[m]})
    raise KeyError(k)
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop and also runs Section 0 if this has not happened in the session, so a stop can also be started on another day or straight from the lecture page.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-09/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: The clinical factors behind the model for coronary artery disease

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 3*

This stop is the mini project of the session. The two models of Week 2 for coronary artery disease, a random forest and a logistic regression, learn on the training patients of the Cleveland data. Their predictions for patient 172, a 59-year-old woman, are shared among her eight clinical variables with exact Shapley values; for the logistic regression, a closed form gives the same values on the scale of the log-odds. Step 1.4 turns the values of every test patient into a global explanation, and step 1.5 asks which of nine factors, among them LDL, age and the ejection fraction, moves a model of the registry of the virtual clinic most.

**At the end of this stop:** The test AUC of the forest (0.904), the Shapley value of the chest pain type of patient 172 in the forest (0.155), the closed-form value of her sex in the logistic regression (−1.175) the mean absolute Shapley value of the chest pain type over the test patients (0.138) and the place of the factor that moves the model of the registry most (9).

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_heart = load_cleveland()
# the split of Week 2, in its order: the training patients first, then the test patients
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _cad_table(order="split"):
    """The 303 patients of the Cleveland data with their set, the eight clinical variables and the disease; in the order of the split of Week 2, or of the patient numbers."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"patient": h.index, "set": np.where(h.index.isin(_cad_tr), "training", "test"),
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "chest pain type": h["cp"].map(_CP),
                         "disease": h["disease"]}).reset_index(drop=True)


print(len(_heart), "patients of the Cleveland data,", int(_heart["disease"].sum()), "with coronary artery disease")
print("Stop 1 is ready.")

### Step 1.1: The two models of Week 2

**Goal.** Fit the two models of Week 2 again on the training patients and judge them on the test patients.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to rebuild the two models for coronary artery disease of an earlier analysis, so that the quality and safety committee can review them before they are used.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit two models on the training patients: a logistic regression with columns standardised on the training patients and at most 1,000 iterations, and a random forest of 300 trees. Fix the random seed to 2026 so that the result can be repeated. Compute the area under the ROC curve (AUC) of both models on the test patients, and the probability that each gives patient 172.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two test AUCs and the two probabilities of patient 172, and keep the test AUC of the forest, with three decimals, under the name "answer".  
> Test: No test patient may be used to fit either model, and the AUC must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])

case = d["patient"] == 172
print("test AUC: forest", round(roc_auc_score(y[te_], forest.predict_proba(X[te_])[:, 1]), 3),
      "| logistic regression", round(roc_auc_score(y[te_], logistic.predict_proba(X[te_])[:, 1]), 3))
print("patient 172: forest", round(forest.predict_proba(X[case])[0, 1], 3),
      "| logistic regression", round(logistic.predict_proba(X[case])[0, 1], 3))
answer = round(float(roc_auc_score(y[te_], forest.predict_proba(X[te_])[:, 1])), 3)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** The forest reaches a test AUC of 0.904. Both models rank the patients well, and they give patient 172 rather different probabilities, which the next steps explain.

### Step 1.2: Shapley values of patient 172

**Goal.** Share the probability that the forest gives patient 172 among her eight clinical variables.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee asks why the forest gives patient 172, a 59-year-old woman with angina on the exercise test, a high probability of coronary artery disease, compared with the average patient.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit a random forest of 300 trees on the training patients. Fix the random seed to 2026 so that the result can be repeated. Treat the eight clinical variables as eight players; the three columns of the chest pain type always move together as one player. For patient 172, compute exact Shapley values of the probability that the forest gives: For each of the 256 groups of players, give the players of the group the values of the patient and the other players the values of each training patient in turn, predict, and average over the training patients; this is the value of the group. The Shapley value of a player is the sum, over all groups without it, of the value of the group with the player minus the value without it, times the weight s!(7 − s)!/8!, where s is the size of the group without the player. Check that the eight values add up to the prediction for the patient minus the value of the empty group.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the eight Shapley values from the largest absolute value to the smallest, with the value of the empty group and the prediction, and keep the Shapley value of the chest pain type, with three decimals, under the name "answer".  
> Test: The eight values must add up to the prediction minus the value of the empty group, to the sixth decimal.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])


def values(model, x, background):
    """Values of the 256 groups for one patient: the mean probability over the background patients."""
    masks = list(itertools.product([0, 1], repeat=8))
    rows = []
    for mask in masks:
        keep = np.zeros(10, dtype=bool)
        for m, cols in zip(mask, players):
            if m:
                keep[cols] = True
        rows.append(np.where(keep, x, background))
    p = model.predict_proba(pd.DataFrame(np.vstack(rows), columns=X.columns))[:, 1]
    return dict(zip(masks, p.reshape(len(masks), len(background)).mean(axis=1)))


def shapley(v):
    phi = np.zeros(8)
    for mask, value in v.items():
        s = sum(mask)
        for j in range(8):
            if mask[j] == 0:
                with_j = mask[:j] + (1,) + mask[j + 1:]
                phi[j] += math.factorial(s) * math.factorial(7 - s) / math.factorial(8) * (v[with_j] - value)
    return phi


background = X[tr_].to_numpy(dtype=float)
names = ["age", "sex", "resting blood pressure", "cholesterol", "maximum heart rate",
               "exercise angina", "ST depression", "chest pain type"]

x = X[d["patient"] == 172].to_numpy(dtype=float)[0]
v = values(forest, x, background)
phi = shapley(v)
print(pd.Series(phi, index=names).sort_values(key=abs, ascending=False).round(3))
print("empty group", round(v[(0,) * 8], 3), "| prediction", round(v[(1,) * 8], 3), "| sum", round(phi.sum(), 6))
answer = round(float(phi[7]), 3)
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** The chest pain type adds 0.155 to the risk of patient 172: In these data a patient without pain at rest who was still sent to angiography often has disease. Being a woman and the absence of ST depression pull her risk down.

### Step 1.3: The closed form of the logistic regression

**Goal.** Compute the Shapley values of the logistic regression for patient 172 on the scale of the log-odds with a closed form.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to see why the logistic regression gives patient 172 a lower probability than the forest, and in which variables the two models differ.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit a logistic regression on the training patients, with columns standardised on the training patients and at most 1,000 iterations. On the scale of the log-odds, the Shapley value of a variable for a patient is its coefficient per unit of the original column times the value of the patient minus the mean of the training patients; for the chest pain type, add the three terms of its columns. Compute the eight values for patient 172 and check that they add up to the log-odds of the patient minus the mean log-odds of the training patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the eight values with the variables, and keep the value of sex, with three decimals, under the name "answer".  
> Test: The eight values must add up to the log-odds of the patient minus the mean log-odds of the training patients.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])

scaler, lr = logistic[0], logistic[1]
beta = lr.coef_[0] / scaler.scale_
x = X[d["patient"] == 172].to_numpy(dtype=float)[0]
terms = beta * (x - X[tr_].mean().to_numpy())
phi = np.array([terms[cols].sum() for cols in players])
names = ["age", "sex", "resting blood pressure", "cholesterol", "maximum heart rate",
               "exercise angina", "ST depression", "chest pain type"]
print(pd.Series(phi, index=names).round(3))
z = logistic.decision_function(X)
print("check:", round(phi.sum(), 6), round(z[d["patient"].to_numpy() == 172][0] - z[tr_].mean(), 6))
answer = round(float(phi[1]), 3)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** Being a woman adds −1.175 to the log-odds of patient 172, the largest of the eight values in size; in the forest, sex ranks only third. The logistic regression gives sex one large coefficient for every patient, while the forest uses sex only in some of its trees.

### Step 1.4: From one patient to all patients

**Goal.** Compute the Shapley values of the forest for every test patient and rank the variables.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know which clinical variables the forest relies on across all test patients, and whether any of them reflects how the patients were referred rather than their disease.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit a random forest of 300 trees on the training patients. Fix the random seed to 2026 so that the result can be repeated. Treat the eight clinical variables as eight players; the three columns of the chest pain type always move together as one player. For every test patient, compute exact Shapley values of the probability that the forest gives: For each of the 256 groups of players, give the players of the group the values of the patient and the other players the values of each training patient in turn, predict, and average over the training patients; this is the value of the group. The Shapley value of a player is the sum, over all groups without it, of the value of the group with the player minus the value without it, times the weight s!(7 − s)!/8!, where s is the size of the group without the player. Check that the eight values add up to the prediction for the patient minus the value of the empty group. Then compute for each variable the mean of the absolute Shapley values over the test patients, and count the test patients in which each variable has the largest absolute value.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the mean absolute values from the largest to the smallest with the counts, and keep the mean absolute value of the chest pain type, with three decimals, under the name "answer".  
> Test: The counts must add up to the number of test patients.  

In [ ]:
#@title Data of step 1.4: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.4
teach.check_step("Step 1.4", "answer", globals())

In [ ]:
#@title Ready code of step 1.4 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])


def values(model, x, background):
    """Values of the 256 groups for one patient: the mean probability over the background patients."""
    masks = list(itertools.product([0, 1], repeat=8))
    rows = []
    for mask in masks:
        keep = np.zeros(10, dtype=bool)
        for m, cols in zip(mask, players):
            if m:
                keep[cols] = True
        rows.append(np.where(keep, x, background))
    p = model.predict_proba(pd.DataFrame(np.vstack(rows), columns=X.columns))[:, 1]
    return dict(zip(masks, p.reshape(len(masks), len(background)).mean(axis=1)))


def shapley(v):
    phi = np.zeros(8)
    for mask, value in v.items():
        s = sum(mask)
        for j in range(8):
            if mask[j] == 0:
                with_j = mask[:j] + (1,) + mask[j + 1:]
                phi[j] += math.factorial(s) * math.factorial(7 - s) / math.factorial(8) * (v[with_j] - value)
    return phi


background = X[tr_].to_numpy(dtype=float)
names = ["age", "sex", "resting blood pressure", "cholesterol", "maximum heart rate",
               "exercise angina", "ST depression", "chest pain type"]

S = np.array([shapley(values(forest, xi, background)) for xi in X[te_].to_numpy(dtype=float)])
table = pd.DataFrame({"mean absolute value": np.abs(S).mean(axis=0),
                            "largest in patients": np.bincount(np.abs(S).argmax(axis=1), minlength=8)}, index=names)
print(table.sort_values("mean absolute value", ascending=False).round(3))
answer = round(float(np.abs(S).mean(axis=0)[7]), 3)
teach.check_step("Step 1.4", "answer", globals())

**What the result shows.** The chest pain type leads with a mean absolute value of 0.138. The model learned that patients without chest pain who were still referred for angiography often have disease; a committee reads this as a feature of the data, not as a cause.

### Step 1.5: Mini project: LDL, age, EF and the model of the registry

**Goal.** Find out which of nine clinical factors moves a model for coronary artery disease most, with exact Shapley values.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know which clinical factor, such as LDL, age or the ejection fraction, influences a model for coronary artery disease most.  
> Data: The notebook holds a table named "data" with the 3,600 patients of the registry of the virtual clinic, a synthetic data set. Each row is a patient: "set" gives "training" (2,700 patients) or "test" (900 patients). The nine clinical factors are "age" (years), "sex" ("man" or "woman"), "smoker", "diabetes", "hypertension" and "family history" (1 when present and 0 otherwise), "LDL" and "HDL" (mg/dL) and "ejection fraction" (%). "coronary artery disease" is 1 when the patient has coronary artery disease and 0 otherwise.  
> Task: Code the sex as 1 for a man and 0 for a woman. Fit a random forest of 200 trees, with at least 10 patients in every leaf, on the training patients to predict coronary artery disease from the nine factors. Fix the random seed to 2026 so that the result can be repeated. Draw 100 training patients as the background and 100 test patients, each with the seed 2026. For each of the 100 test patients, compute exact Shapley values of the probability of the forest, with the nine factors as nine players: For each of the 512 groups of players, give the players of the group the values of the patient and the other players the values of each background patient in turn, predict, and average over the background; this is the value of the group. The Shapley value of a player is the sum, over all groups without it, of the value of the group with the player minus the value without it, times the weight s!(8 − s)!/9!, where s is the size of the group without the player. Then compute for each factor the mean of the absolute Shapley values over the 100 test patients.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the nine mean absolute values from the largest to the smallest, and keep under the name "answer" the place of the factor with the largest value in the list of the factors above, counted from 1: age 1, sex 2 and so on up to the ejection fraction, 9.  
> Test: The Shapley values of every test patient must add up to the probability of the forest for the patient minus the value of the empty group.  

In [ ]:
#@title Data of step 1.5: run this cell (it prepares the table "data" and shows it)
from vclinic import make_cohort
_reg = make_cohort(3600)
_rtr, _rte = train_test_split(np.arange(len(_reg)), test_size=0.25, stratify=_reg["cad"], random_state=2026)
_set = np.where(np.isin(np.arange(len(_reg)), _rtr), "training", "test")
data = pd.DataFrame({"set": _set, "age": _reg["age"], "sex": np.where(_reg["sex"] == "M", "man", "woman"),
                            "smoker": _reg["smoker"], "diabetes": _reg["diabetes"], "hypertension": _reg["hypertension"],
                            "family history": _reg["family_history"], "LDL": _reg["ldl"], "HDL": _reg["hdl"],
                            "ejection fraction": _reg["lvef"], "coronary artery disease": _reg["cad"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.5
teach.check_step("Step 1.5", "answer", globals())

In [ ]:
#@title Ready code of step 1.5 (run it only if the code of the assistant does not pass the check)
from vclinic import make_cohort
_reg = make_cohort(3600)
_rtr, _rte = train_test_split(np.arange(len(_reg)), test_size=0.25, stratify=_reg["cad"], random_state=2026)
_set = np.where(np.isin(np.arange(len(_reg)), _rtr), "training", "test")
data = pd.DataFrame({"set": _set, "age": _reg["age"], "sex": np.where(_reg["sex"] == "M", "man", "woman"),
                            "smoker": _reg["smoker"], "diabetes": _reg["diabetes"], "hypertension": _reg["hypertension"],
                            "family history": _reg["family_history"], "LDL": _reg["ldl"], "HDL": _reg["hdl"],
                            "ejection fraction": _reg["lvef"], "coronary artery disease": _reg["cad"]})
factors = ["age", "sex", "smoker", "diabetes", "hypertension", "family history", "LDL", "HDL", "ejection fraction"]
X = data[factors].assign(**{"sex": (data["sex"] == "man").astype(int)})
y = data["coronary artery disease"].to_numpy()
train = (data["set"] == "training").to_numpy()
forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=2026).fit(X[train], y[train])
background = X[train].sample(100, random_state=2026).to_numpy(dtype=float)
patients = X[~train].sample(100, random_state=2026).to_numpy(dtype=float)
masks = list(itertools.product([0, 1], repeat=9))
S = []
for x in patients:
    rows = np.vstack([np.where(np.array(mask, dtype=bool), x, background) for mask in masks])
    v = dict(zip(masks, forest.predict_proba(pd.DataFrame(rows, columns=X.columns))[:, 1].reshape(len(masks), -1).mean(axis=1)))
    phi = np.zeros(9)
    for mask, value in v.items():
        s = sum(mask)
        for j in range(9):
            if mask[j] == 0:
                phi[j] += math.factorial(s) * math.factorial(8 - s) / math.factorial(9) * (v[mask[:j] + (1,) + mask[j + 1:]] - value)
    S.append(phi)
ranking = pd.Series(np.abs(np.array(S)).mean(axis=0), index=factors)
print(ranking.sort_values(ascending=False).round(3))
answer = int(np.argmax(ranking.to_numpy())) + 1
teach.check_step("Step 1.5", "answer", globals())

**What the result shows.** The ejection fraction, factor 9, moves the forest most. The virtual clinic never used it to draw the disease; it lowered the ejection fraction of the patients who already had coronary artery disease. A Shapley value says what the model uses, not what causes the disease.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#lime-a-simple-model-around-one-patient) and continue reading at «LIME: Local explanations». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: LIME for one patient, and where the echo network looks

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 4 to 7*

This stop explains predictions in two more ways. LIME imitates the forest of Stop 1 around patient 172 with a weighted linear model, and a repetition with twenty seeds shows how stable its answer is. A small convolutional network then learns to flag an ejection fraction below 45 percent on echocardiograms of CAMUS, and its Grad-CAM maps are compared with the expert outlines of the left ventricle.

**At the end of this stop:** The LIME coefficient of the chest pain type for patient 172 (0.145), the share of twenty seeds that put the chest pain type first with only 100 neighbours (0.70), the test AUC of the echo network (0.796) and the mean share of the Grad-CAM maps inside the expert outline (0.317).

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_heart = load_cleveland()
# the split of Week 2, in its order: the training patients first, then the test patients
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _cad_table(order="split"):
    """The 303 patients of the Cleveland data with their set, the eight clinical variables and the disease; in the order of the split of Week 2, or of the patient numbers."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"patient": h.index, "set": np.where(h.index.isin(_cad_tr), "training", "test"),
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "chest pain type": h["cp"].map(_CP),
                         "disease": h["disease"]}).reset_index(drop=True)

_SETS_E = {"training": "training", "validation": "validation", "testing": "test"}


def _echo():
    """The views of CAMUS of the course with their frames and expert outlines (loaded once, a few seconds)."""
    if "_echo_cache" not in globals():
        frames_ = camus.frames(camus.patients(), size=96)
        info = camus.table()
        low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
        index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
        views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 patients x 2 views
        echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
        labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
        outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # cavity or myocardium at ED or ES
        echo_y = np.array([low_ef[p] for p, v in views])
        echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
        etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"
        globals()["_echo_cache"] = dict(views=views, echo_x=echo_x, outline=outline, echo_y=echo_y, echo_set=echo_set,
                                        quality=camus.table().set_index("patient"))
    return _echo_cache


def _echo_table():
    """The 600 views of the 300 patients of CAMUS of the course, with their frames, the expert outline and an ejection fraction below 45 percent."""
    e = _echo()
    q = e["quality"]
    return pd.DataFrame({"patient": [p for p, v in e["views"]], "view": [v for p, v in e["views"]],
                         "set": [_SETS_E[s] for s in e["echo_set"]],
                         "image quality": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in e["views"]],
                         "end-diastole": list(e["echo_x"][:, 0]), "end-systole": list(e["echo_x"][:, 1]),
                         "outline": list(e["outline"].astype(np.uint8)), "low ejection fraction": e["echo_y"]})


print(len(_heart), "patients of the Cleveland data and", len(_echo()["views"]), "views of CAMUS")
print("Stop 2 is ready.")

### Step 2.1: LIME for patient 172

**Goal.** Explain the probability that the forest gives patient 172 with a weighted linear model fitted around her.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants a second explanation of the prediction for patient 172, made in another way than the Shapley values, to see whether the two agree.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit a random forest of 300 trees on the training patients. Fix the random seed to 2026 so that the result can be repeated. Treat the eight clinical variables as eight players; the three columns of the chest pain type always move together as one player. Explain the probability that the forest gives patient 172 with LIME. Draw the neighbours of the patient: In each neighbour, draw one training patient at random, and let each of the eight players keep the value of patient 172 with a probability of one half or otherwise take the value of the drawn training patient. Record for each neighbour which players kept the value of the patient (1) and which were replaced (0). Weight each neighbour by exp(−d²/w²), where d is the square root of the number of players replaced and w = 0.75 × √8. Fit a ridge regression with a penalty of 1 of the probabilities of the forest for the neighbours on the eight columns of 0 and 1, with these weights; its eight coefficients are the explanation. Draw first the 0 and 1 of all neighbours and then their training patients, from one random generator. Use 2,000 neighbours and the seed 2026.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the eight coefficients from the largest absolute value to the smallest, and keep the coefficient of the chest pain type, with three decimals, under the name "answer".  
> Test: The weight of a neighbour that keeps all eight values of the patient must be 1.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])

background = X[tr_].to_numpy(dtype=float)

def lime(n, seed):
    rng = np.random.default_rng(seed)
    Z = rng.integers(0, 2, size=(n, 8))
    rows = background[rng.integers(0, len(background), size=n)].copy()
    for j, cols in enumerate(players):
        rows[:, cols] = np.where(Z[:, [j]] == 1, x[cols], rows[:, cols])
    p = forest.predict_proba(pd.DataFrame(rows, columns=X.columns))[:, 1]
    d2 = (1 - Z).sum(axis=1)
    w = np.exp(-d2 / (0.75 * np.sqrt(8)) ** 2)
    return Ridge(alpha=1.0).fit(Z, p, sample_weight=w).coef_


x = X[d["patient"] == 172].to_numpy(dtype=float)[0]

names = ["age", "sex", "resting blood pressure", "cholesterol", "maximum heart rate",
               "exercise angina", "ST depression", "chest pain type"]
coef = lime(2000, RANDOM_STATE)
print(pd.Series(coef, index=names).sort_values(key=abs, ascending=False).round(3))
answer = round(float(coef[7]), 3)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** LIME gives the chest pain type a coefficient of 0.145, close to its Shapley value in Stop 1. With neighbours that keep or replace the values of the patient, LIME and the Shapley values answer almost the same question.

### Step 2.2: How stable is LIME?

**Goal.** Repeat LIME with few neighbours and twenty seeds, and see how often the leading variable changes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know whether an explanation by LIME would stay the same if it were run again, before it trusts an explanation of a single patient.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain, with typical angina as the reference. Keep the rows in the order of the table. Fit a random forest of 300 trees on the training patients. Fix the random seed to 2026 so that the result can be repeated. Treat the eight clinical variables as eight players; the three columns of the chest pain type always move together as one player. Explain the probability that the forest gives patient 172 with LIME. Draw the neighbours of the patient: In each neighbour, draw one training patient at random, and let each of the eight players keep the value of patient 172 with a probability of one half or otherwise take the value of the drawn training patient. Record for each neighbour which players kept the value of the patient (1) and which were replaced (0). Weight each neighbour by exp(−d²/w²), where d is the square root of the number of players replaced and w = 0.75 × √8. Fit a ridge regression with a penalty of 1 of the probabilities of the forest for the neighbours on the eight columns of 0 and 1, with these weights; its eight coefficients are the explanation. Draw first the 0 and 1 of all neighbours and then their training patients, from one random generator. Use only 100 neighbours, and repeat with the seeds 1 to 20. For each seed, find the player with the largest absolute coefficient.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show how often each player comes first, and the smallest and largest coefficient of the chest pain type, and keep the share of seeds in which the chest pain type comes first, with two decimals, under the name "answer".  
> Test: The counts of the players must add up to 20.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
data = _cad_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
data = _cad_table()
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[tr_], y[tr_])
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[tr_], y[tr_])

background = X[tr_].to_numpy(dtype=float)

def lime(n, seed):
    rng = np.random.default_rng(seed)
    Z = rng.integers(0, 2, size=(n, 8))
    rows = background[rng.integers(0, len(background), size=n)].copy()
    for j, cols in enumerate(players):
        rows[:, cols] = np.where(Z[:, [j]] == 1, x[cols], rows[:, cols])
    p = forest.predict_proba(pd.DataFrame(rows, columns=X.columns))[:, 1]
    d2 = (1 - Z).sum(axis=1)
    w = np.exp(-d2 / (0.75 * np.sqrt(8)) ** 2)
    return Ridge(alpha=1.0).fit(Z, p, sample_weight=w).coef_


x = X[d["patient"] == 172].to_numpy(dtype=float)[0]

names = ["age", "sex", "resting blood pressure", "cholesterol", "maximum heart rate",
               "exercise angina", "ST depression", "chest pain type"]
runs = np.array([lime(100, s) for s in range(1, 21)])
first = [names[j] for j in np.abs(runs).argmax(axis=1)]
print(pd.Series(first).value_counts())
print("chest pain type from", round(runs[:, 7].min(), 3), "to", round(runs[:, 7].max(), 3))
answer = round(float(np.mean([f == names[7] for f in first])), 2)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** With 100 neighbours the chest pain type comes first in a share of 0.70 of the seeds, and the others put exercise angina or another variable first. A report of LIME states the number of neighbours and shows that the result holds over seeds.

### Step 2.3: A network for a low ejection fraction

**Goal.** Train a small convolutional network that flags an ejection fraction below 45 percent on one view.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee reviews a network that reads echocardiograms. Before its maps are judged, the cardiologist wants to know how well it flags a low ejection fraction on views of patients it has not seen.  
> Data: The notebook holds a table named "data" with 600 views of CAMUS, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 300 patients. Each row is a view: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients) and "image quality" the grade of the experts ("Good", "Medium" or "Poor"). Each cell of "end-diastole" and of "end-systole" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of "outline" the expert outline of the left ventricular cavity and the myocardium, at end-diastole or end-systole, at the same size: 1 inside and 0 outside. "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent and 0 otherwise.  
> Task: Build a small convolutional network for one view that takes the end-diastolic and the end-systolic frame as two channels of 96 × 96 pixels. It has four blocks, and each block is a convolution with 3 × 3 kernels that keeps the size (16, 32, 64 and 64 channels), batch normalisation, ReLU and a 2 × 2 max pooling. After the blocks, the network takes the mean of every channel over the image and applies a dropout of 0.3, and a linear layer with one output gives the logit of a low ejection fraction. Train it on the training views with Adam (learning rate 0.001, weight decay 0.0001) and the binary cross-entropy for 30 epochs in shuffled groups of 32 views, shifting each group at random by up to 4 pixels in each direction, and keep the weights of the epoch with the best AUC on the validation views. Seed the random generator of the network library with 2026 before building the network and again before training. Compute the AUC on the test views.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the best validation AUC and the test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: No test view may be used to choose the epoch, and the views of one patient must stay in one set.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
data = _echo_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
data = _echo_table()
d = data
Xe = np.stack([np.stack(d["end-diastole"]), np.stack(d["end-systole"])], axis=1).astype(np.float32)
ye = d["low ejection fraction"].to_numpy()
etr_, eva_, ete_ = [(d["set"] == s).to_numpy() for s in ("training", "validation", "test")]


class EchoNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        c = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(c[i], c[i + 1], 3, padding=1), nn.BatchNorm2d(c[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


@torch.no_grad()
def predict(net, X):
    net.eval()
    return torch.cat([torch.sigmoid(net(torch.as_tensor(X[i:i + 64]))) for i in range(0, len(X), 64)]).numpy()


def train():
    torch.manual_seed(RANDOM_STATE)
    net = EchoNetwork()
    torch.manual_seed(RANDOM_STATE)
    Xt, yt = torch.as_tensor(Xe[etr_]), torch.as_tensor(ye[etr_], dtype=torch.float32)
    opt = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(30):
        net.train()
        order = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 32):
            b = order[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            loss = loss_fn(net(torch.roll(Xt[b], shifts=(dy, dx), dims=(2, 3))), yt[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
        a = roc_auc_score(ye[eva_], predict(net, Xe[eva_]))
        if a > best:
            best, state = a, copy.deepcopy(net.state_dict())
    net.load_state_dict(state)
    return net

net = train()
p_test = predict(net, Xe[ete_])
print("test AUC", round(roc_auc_score(ye[ete_], p_test), 3))
answer = round(float(roc_auc_score(ye[ete_], p_test)), 3)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** The network reaches a test AUC of 0.796 on single views: It flags many low ejection fractions, far from the method of discs of Week 4. Its maps are judged in the next step.

### Step 2.4: Grad-CAM maps against the expert outline

**Goal.** Compute the Grad-CAM map of every test view and the share of the map inside the expert outline.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know whether the network flags a low ejection fraction from the left ventricle, where the expert looks, or from other parts of the image.  
> Data: The notebook holds a table named "data" with 600 views of CAMUS, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 300 patients. Each row is a view: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients) and "image quality" the grade of the experts ("Good", "Medium" or "Poor"). Each cell of "end-diastole" and of "end-systole" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of "outline" the expert outline of the left ventricular cavity and the myocardium, at end-diastole or end-systole, at the same size: 1 inside and 0 outside. "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent and 0 otherwise.  
> Task: Build a small convolutional network for one view that takes the end-diastolic and the end-systolic frame as two channels of 96 × 96 pixels. It has four blocks, and each block is a convolution with 3 × 3 kernels that keeps the size (16, 32, 64 and 64 channels), batch normalisation, ReLU and a 2 × 2 max pooling. After the blocks, the network takes the mean of every channel over the image and applies a dropout of 0.3, and a linear layer with one output gives the logit of a low ejection fraction. Train it on the training views with Adam (learning rate 0.001, weight decay 0.0001) and the binary cross-entropy for 30 epochs in shuffled groups of 32 views, shifting each group at random by up to 4 pixels in each direction, and keep the weights of the epoch with the best AUC on the validation views. Seed the random generator of the network library with 2026 before building the network and again before training. For every test view, compute the Grad-CAM map of the output at the ReLU of the last block, before its pooling. To do so, take the gradient of the output with respect to the 64 feature maps of 12 × 12 at that layer, use the mean gradient of each channel as its weight and form the weighted sum of the 64 maps. Keep only the positive part of the sum and enlarge it to 96 × 96 by bilinear interpolation. For each view, compute the share of the sum of the map that falls inside the outline. Report the mean of the shares over the test views and, for comparison, the mean share of the image that the outline covers.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two means, and keep the mean share of the maps inside the outline, with three decimals, under the name "answer".  
> Test: Every share must lie between 0 and 1, and no map may be computed from a network in training mode.  

In [ ]:
#@title Data of step 2.4: run this cell (it prepares the table "data" and shows it)
data = _echo_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.4
teach.check_step("Step 2.4", "answer", globals())

In [ ]:
#@title Ready code of step 2.4 (run it only if the code of the assistant does not pass the check)
data = _echo_table()
d = data
Xe = np.stack([np.stack(d["end-diastole"]), np.stack(d["end-systole"])], axis=1).astype(np.float32)
ye = d["low ejection fraction"].to_numpy()
etr_, eva_, ete_ = [(d["set"] == s).to_numpy() for s in ("training", "validation", "test")]


class EchoNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        c = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(c[i], c[i + 1], 3, padding=1), nn.BatchNorm2d(c[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


@torch.no_grad()
def predict(net, X):
    net.eval()
    return torch.cat([torch.sigmoid(net(torch.as_tensor(X[i:i + 64]))) for i in range(0, len(X), 64)]).numpy()


def train():
    torch.manual_seed(RANDOM_STATE)
    net = EchoNetwork()
    torch.manual_seed(RANDOM_STATE)
    Xt, yt = torch.as_tensor(Xe[etr_]), torch.as_tensor(ye[etr_], dtype=torch.float32)
    opt = torch.optim.Adam(net.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(30):
        net.train()
        order = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 32):
            b = order[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            loss = loss_fn(net(torch.roll(Xt[b], shifts=(dy, dx), dims=(2, 3))), yt[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
        a = roc_auc_score(ye[eva_], predict(net, Xe[eva_]))
        if a > best:
            best, state = a, copy.deepcopy(net.state_dict())
    net.load_state_dict(state)
    return net


def maps(net, X):
    net.eval()
    saved = {}
    h = net.blocks[3][2].register_forward_hook(lambda m, i, o: saved.update(a=o))
    out = net(torch.as_tensor(X))
    h.remove()
    g = torch.autograd.grad(out.sum(), saved["a"])[0]
    cam = torch.relu((g.mean(dim=(2, 3), keepdim=True) * saved["a"]).sum(dim=1, keepdim=True))
    return torch.nn.functional.interpolate(cam, size=(96, 96), mode="bilinear", align_corners=False)[:, 0].detach().numpy()

net = train()
cams = maps(net, Xe[ete_])
mask = np.stack(d["outline"])[ete_].astype(float)
share = (cams * mask).sum(axis=(1, 2)) / np.maximum(cams.sum(axis=(1, 2)), 1e-12)
print("share of the map inside the outline", round(share.mean(), 3), "| area of the outline", round(mask.mean(), 3))
answer = round(float(share.mean()), 3)
teach.check_step("Step 2.4", "answer", globals())

**What the result shows.** On average a share of 0.317 of a map falls inside outlines that cover about a fifth of the image: The maps prefer the heart, though far from only the heart. Whether this is about the model is the question of the sanity check in Section 7 and in the weekly project.

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#last-two-hours-ai-reliability-and-clinical-validation) and continue reading at «Last two hours: AI reliability and clinical validation». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Shift, monitoring and subgroups

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 8 to 10*

This stop asks whether a model stays reliable. A classifier learns to tell the presentations of hospital B from those of hospital A in the chest pain unit of Week 8, before any outcome is known. The observed and expected events at B are then followed in the order of arrival. The risk model is judged in groups of sex and age at a threshold for discharge, and the model for coronary artery disease is fitted with and without sex.

**At the end of this stop:** The AUC of the classifier that tells the hospitals apart (0.988), the presentation at which observed minus expected events first leave their band at B (253), the sensitivity in women at the threshold for discharge (0.956) and the mean predicted risk of the women of the Cleveland data from a model without sex (0.417).

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_heart = load_cleveland()
# the split of Week 2, in its order: the training patients first, then the test patients
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "typical angina", 2.0: "atypical angina", 3.0: "non-anginal pain", 4.0: "no pain"}


def _cad_table(order="split"):
    """The 303 patients of the Cleveland data with their set, the eight clinical variables and the disease; in the order of the split of Week 2, or of the patient numbers."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"patient": h.index, "set": np.where(h.index.isin(_cad_tr), "training", "test"),
                         "age": h["age"].astype(int), "sex": h["sex"].map({1.0: "man", 0.0: "woman"}),
                         "resting blood pressure": h["trestbps"].astype(int), "cholesterol": h["chol"].astype(int),
                         "maximum heart rate": h["thalach"].astype(int), "exercise angina": h["exang"].astype(int),
                         "ST depression": h["oldpeak"], "chest pain type": h["cp"].map(_CP),
                         "disease": h["disease"]}).reset_index(drop=True)



def _unit():
    """The presentations of hospitals A and B and the probabilities of MACE from the risk model of the chest pain unit (computed once, a few seconds)."""
    if "_unit_cache" not in globals():
        def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
            """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
            best = None
            for C in (Cs if Xv is not None else (1.0,)):
                model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
                auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
                if best is None or auc > best[1]:
                    best = (model, auc)
            return best[0]


        def logit(p):
            """The log-odds of a probability."""
            p = np.clip(p, 1e-6, 1 - 1e-6)
            return np.log(p / (1 - p))


        def bootstrap_auc(y, p, n=2000, seed=2026):
            """AUC in n bootstrap samples of the patients: an array of n values."""
            rng = np.random.default_rng(seed)
            out = []
            while len(out) < n:
                i = rng.integers(0, len(y), len(y))
                if 0 < y[i].sum() < len(i):
                    out.append(roc_auc_score(y[i], p[i]))
            return np.array(out)


        def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
            """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
            rng = np.random.default_rng(seed)
            out = []
            while len(out) < n:
                i = rng.integers(0, len(y), len(y))
                if 0 < y[i].sum() < len(i):
                    out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
            return np.array(out)


        def calibration(y, p):
            """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
            z = logit(p)
            a = 0.0
            for _ in range(50):
                q = 1 / (1 + np.exp(-(a + z)))
                a += (y - q).sum() / (q * (1 - q)).sum()
            slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
            return a, slope


        def reliability(y, p, bins=10):
            """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
            order = np.argsort(p)
            groups = np.array_split(order, bins)
            return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])


        def share_threshold(p, k):
            """The threshold that sends home the k patients with the lowest predicted risk: the k-th lowest risk."""
            return float(np.sort(p)[k - 1])


        def rule_out_threshold(y, p, sensitivity):
            """The largest threshold that keeps the sensitivity for the events at least at the given level. Patients at or below it are discharged. It is chosen on the validation set."""
            order = np.argsort(p)
            missed = np.cumsum(y[order])              # events at or below each threshold
            ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
            return float(p[order][ok.max()]) if len(ok) else 0.0


        def wilson(k, n, z=1.96):
            """The 95 percent Wilson interval of a proportion k/n."""
            if n == 0:
                return (0.0, 1.0)
            p = k / n
            c = (p + z * z / (2 * n)) / (1 + z * z / n)
            h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
            return (c - h, c + h)


        def net_benefit(y, p, thresholds):
            """Net benefit of treating the patients over each threshold: true positives minus false positives weighted by the odds of the threshold, per patient."""
            n = len(y)
            return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])

        chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
        A = chest[chest["hospital"] == "A"].reset_index(drop=True)
        B = chest[chest["hospital"] == "B"].reset_index(drop=True)
        tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
        y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

        CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                    "heart_rate", "sbp", "egfr"]


        def clinical(d):
            """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
            return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


        def laboratory(d):
            """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
            url = d["troponin_url"]
            t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
            return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                                 "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


        def echo(d):
            """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
            return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
        svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


        def note(d):
            """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
            return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

        inputs = lambda d: pd.concat([clinical(d), laboratory(d), echo(d), note(d)], axis=1)
        XA, XB = inputs(A), inputs(B)
        unit_model = fit_logistic(XA[tr], y[tr], Xv=XA[va], yv=y[va])
        pA, pB = unit_model.predict_proba(XA)[:, 1], unit_model.predict_proba(XB)[:, 1]
        globals()["_unit_cache"] = dict(A=A, B=B, pA=pA, pB=pB)
    return _unit_cache


_A, _B = _unit()["A"], _unit()["B"]
_va, _te = [(_A["set"] == _s).to_numpy() for _s in ("validation", "test")]
_SETS_U = {"training": "training", "validation": "validation", "test": "test"}


def _inputs_table():
    """The presentations of hospitals A and B with what the team knows at arrival."""
    rows = []
    for name, d in (("A", _A), ("B", _B)):
        url = d["troponin_url"]
        rows.append(pd.DataFrame({"hospital": name, "set": d["set"].map(_SETS_U).fillna("hospital B") if name == "A" else "hospital B",
                                  "age": d["age"], "sex": d["sex"].map({"M": "man", "F": "woman"}),
                                  "smoker": d["smoker"], "diabetes": d["diabetes"], "hypertension": d["hypertension"],
                                  "hyperlipidaemia": d["hyperlipidaemia"], "obesity": d["obesity"],
                                  "family history": d["family_history"], "known coronary artery disease": d["known_cad"],
                                  "heart rate": d["heart_rate"], "systolic pressure": d["sbp"], "eGFR": d["egfr"],
                                  "troponin at arrival": (d["troponin_0h"] / url).round(2),
                                  "troponin after one hour": (d["troponin_1h"] / url).round(2),
                                  "echocardiogram done": d["echo_done"], "ejection fraction": d["lvef"],
                                  "wall motion abnormality": d["rwma"], "triage note": d["note_tr"]}))
    return pd.concat(rows, ignore_index=True)


print(len(_A), "presentations at hospital A and", len(_B), "at hospital B")
print("Stop 3 is ready.")

### Step 3.1: Can the two hospitals be told apart?

**Goal.** Train a classifier that tells the presentations of hospital B from those of hospital A by their inputs, without any outcome.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: A risk model of the chest pain unit of hospital A is to be used at hospital B. Before the first outcomes arrive, the committee wants to know whether the patients and the records of B look like those of A, and where they differ.  
> Data: The notebook holds a table named "data" with the presentations with chest pain of two hospitals of the virtual clinic, a synthetic data set: 4,000 at hospital A and 1,000 at hospital B. Each row is a presentation: "hospital" gives the hospital ("A" or "B") and "set" its set ("training", "validation" and "test" at A; "hospital B" at B). The clinical table gives "age" (years), "sex" ("man" or "woman"), seven risk factors that are 1 when present ("smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family history" and "known coronary artery disease"), "heart rate" (per minute), "systolic pressure" (mmHg) and "eGFR". Troponin is given in multiples of the upper reference limit of the assay of each hospital: "troponin at arrival" and "troponin after one hour", which is empty when it was not measured. The bedside echocardiogram gives "echocardiogram done" (1 or 0), and when it was done "ejection fraction" (%) and "wall motion abnormality" (1 when a regional abnormality was seen); both are empty otherwise. "triage note" is the note of the triage nurse in Turkish.  
> Task: Build the inputs of a risk model for MACE for every presentation: The clinical table (age, sex as 1 for a man, the seven risk factors, heart rate, systolic pressure, eGFR); the troponin (the logarithm of the troponin at arrival, the logarithm of the troponin after one hour with the value at arrival when it was not measured, the change in one hour with 0 when it was not measured, and 1 when the second troponin was measured); the echocardiogram (echocardiogram done, the ejection fraction with 60 when it was not done, the wall motion abnormality with 0 when it was not done); and the note as 20 numbers (TF-IDF of words and word pairs that occur in at least 3 notes, with sublinear term frequency, fitted on the training notes of A and reduced to 20 components by a singular value decomposition fitted on the same notes, with the seed 2026). Then fit a logistic regression with standardised columns and at most 5,000 iterations that tells hospital B (1) from hospital A (0), with the predictions from a five-fold stratified cross-validation of all 5,000 presentations, shuffled with the seed 2026, and compute its AUC. Repeat with the columns of each group of inputs alone.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the AUC for all inputs and for each group, and keep the AUC for all inputs, with three decimals, under the name "answer".  
> Test: No outcome may be used, and the note model may learn from the training notes of A only.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
data = _inputs_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
data = _inputs_table()
d = data
t0, t1 = d["troponin at arrival"], d["troponin after one hour"]
groups = {
    "clinical": pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                                   **{c: d[c] for c in ["smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity",
                                                                  "family history", "known coronary artery disease", "heart rate",
                                                                  "systolic pressure", "eGFR"]}}),
    "troponin": pd.DataFrame({"log0": np.log(t0), "log1": np.log(t1.fillna(t0)), "change": (t1 - t0).fillna(0.0),
                              "second": t1.notna().astype(int)}),
    "echocardiogram": pd.DataFrame({"done": d["echocardiogram done"],
                                                    "ejection fraction": d["ejection fraction"].fillna(60.0),
                                                    "wall motion": d["wall motion abnormality"].fillna(0.0)})}
train_notes = d.loc[d["set"] == "training", "triage note"]
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(train_notes)
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(train_notes))
groups["note"] = pd.DataFrame(svd.transform(tfidf.transform(d["triage note"])))
hosp = (d["hospital"] == "B").astype(int).to_numpy()
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


def auc_of(X):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, hosp, cv=folds, method="predict_proba")[:, 1]
    return roc_auc_score(hosp, p)


all_ = pd.concat(list(groups.values()), axis=1)
all_.columns = [str(c) for c in all_.columns]
print("all inputs", round(auc_of(all_), 3))
for name, X in groups.items():
    print(name, round(auc_of(X), 3))
answer = round(float(auc_of(all_)), 3)
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** The two hospitals are told apart with an AUC of 0.988, almost entirely by the notes: The nurses of B write shorter notes. The classifier shows what changed in the inputs, not whether the risk model still works.

### Step 3.2: Observed against expected events at hospital B

**Goal.** Follow observed minus expected events at hospital B in the order of arrival and find the first alarm.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The model of hospital A is now used at hospital B, and the outcomes of its patients arrive one after the other. The committee wants a simple chart that raises an alarm when the model expects too few or too many events.  
> Data: The notebook holds a table named "data" with the 1,000 presentations with chest pain of a second hospital, B, of the virtual clinic, a synthetic data set, in the order in which they arrived. Each row is a presentation: "order" gives its place in time, "probability" the probability of a major adverse cardiac event within 30 days from a model that learned at the first hospital, A, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Go through the presentations in their order. After each presentation m, compute the sum of MACE minus the sum of the probabilities up to m, and the band 1.96 times the square root of the sum of p(1 − p) up to m. Find the first presentation at which the difference lies above the band, and draw the difference with the band against the order.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the chart, print the difference and the band after all presentations, and keep the order of the first presentation above the band under the name "answer".  
> Test: The band must start near 0 and widen with the number of presentations.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
data = pd.DataFrame({"order": np.arange(1, len(_B) + 1), "probability": _unit()["pB"], "MACE": _B["mace30"].to_numpy()})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"order": np.arange(1, len(_B) + 1), "probability": _unit()["pB"], "MACE": _B["mace30"].to_numpy()})
d = data.sort_values("order")
p, y = d["probability"].to_numpy(), d["MACE"].to_numpy()
diff = np.cumsum(y - p)
band = 1.96 * np.sqrt(np.cumsum(p * (1 - p)))
above = np.where(diff > band)[0]
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.plot(d["order"], diff, color="#9B1B1B")
ax.fill_between(d["order"], -band, band, color="#1F5F8B", alpha=0.15)
ax.set_xlabel("order of arrival")
ax.set_ylabel("observed − expected")
plt.show()
print("after all:", round(diff[-1], 1), "band ±", round(band[-1], 1))
answer = int(d["order"].to_numpy()[above[0]]) if len(above) else 0
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** The curve leaves its band for the first time at presentation 253, and after all 1,000 presentations it lies far above it: The model of A expects too few events at B.

### Step 3.3: Subgroups at a threshold for discharge

**Goal.** Choose a threshold for discharge on the validation presentations and judge it on the test presentations by sex and age.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know whether a threshold for discharge that is safe on average is also safe for women, for men, for patients under 65 and for patients aged 65 or more.  
> Data: The notebook holds a table named "data" with the 800 validation and the 800 test presentations of a chest pain unit of the virtual clinic, a synthetic data set. Each row is a presentation: "set" gives its set ("validation" or "test"), "sex" the sex ("man" or "woman"), "age" the age in years, "probability" the probability of a major adverse cardiac event within 30 days from a model of the course that learned on the training presentations, and "MACE" is 1 when such an event followed and 0 otherwise.  
> Task: Choose the threshold on the validation presentations only: the largest probability at which at most 2 percent of the validation presentations with MACE fall at or below it. Presentations at or below the threshold are sent home. On the test presentations, for all, for women, for men, for patients under 65 and for patients aged 65 or more, compute the number of presentations and of MACE, the area under the ROC curve (AUC), the mean probability, the observed share of MACE, the number sent home, the MACE among them and the sensitivity.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each group, and keep the sensitivity in women, with three decimals, under the name "answer".  
> Test: The threshold must come from the validation presentations only.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k], "sex": _A["sex"].map({"M": "man", "F": "woman"}).to_numpy()[_k],
                     "age": _A["age"].to_numpy()[_k], "probability": _unit()["pA"][_k], "MACE": _A["mace30"].to_numpy()[_k]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
_k = _va | _te
data = pd.DataFrame({"set": np.where(_va, "validation", "test")[_k], "sex": _A["sex"].map({"M": "man", "F": "woman"}).to_numpy()[_k],
                     "age": _A["age"].to_numpy()[_k], "probability": _unit()["pA"][_k], "MACE": _A["mace30"].to_numpy()[_k]})
d = data
va_, te_ = (d["set"] == "validation").to_numpy(), (d["set"] == "test").to_numpy()
p, y = d["probability"].to_numpy(), d["MACE"].to_numpy()
order = np.argsort(p[va_])
missed = np.cumsum(y[va_][order])
ok = np.where(missed <= 0.02 * y[va_].sum())[0]
thr = float(p[va_][order][ok.max()])
groups = {"all": np.ones(len(d), bool), "women": (d["sex"] == "woman").to_numpy(),
                  "men": (d["sex"] == "man").to_numpy(), "under 65": (d["age"] < 65).to_numpy(),
                  "65 or older": (d["age"] >= 65).to_numpy()}
rows = []
for name, g in groups.items():
    m = g & te_
    home = p[m] <= thr
    rows.append({"group": name, "n": int(m.sum()), "MACE": int(y[m].sum()), "AUC": round(roc_auc_score(y[m], p[m]), 3),
                         "mean probability": round(p[m].mean(), 3), "observed": round(y[m].mean(), 3),
                         "sent home": int(home.sum()), "MACE among them": int(y[m][home].sum()),
                         "sensitivity": round(1 - y[m][home].sum() / y[m].sum(), 3)})
table = pd.DataFrame(rows)
print("threshold", round(thr, 4))
print(table.to_string(index=False))
answer = float(table.iloc[1]["sensitivity"])
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** The threshold keeps a sensitivity of 98 percent on the validation presentations, but the test women reach 0.956, and both missed events are younger women. With few events in each group, the committee asks for the difference to be watched rather than taken as proof.

### Step 3.4: A model without sex

**Goal.** Fit the logistic regression of Week 2 with and without sex and compare its calibration and its errors in women and men.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Someone proposes to remove sex from the model for coronary artery disease so that it treats women and men alike. The committee wants to know what this would do to the predictions for women and for men.  
> Data: The notebook holds a table named "data" with the 303 patients of the Cleveland data of the UCI Heart Disease data set, patients referred for coronary angiography. Each row is a patient: "patient" gives the number of the patient in the data (0 to 302) and "set" the set of Week 2 ("training", 227 patients; "test", 76 patients). The clinical variables are "age" (years), "sex" ("man" or "woman"), "resting blood pressure" (mmHg), "cholesterol" (mg/dL), "maximum heart rate" (the highest heart rate during the exercise test, per minute), "exercise angina" (1 when the exercise test brought on angina), "ST depression" (the depression of the ST segment during exercise compared with rest, mm) and "chest pain type" ("typical angina", "atypical angina", "non-anginal pain" or "no pain"). "disease" is 1 when the angiogram showed coronary artery disease and 0 otherwise.  
> Task: Turn the table into the columns of a model, in this order: age, sex as 1 for a man and 0 for a woman, resting blood pressure, cholesterol, maximum heart rate, exercise angina, ST depression, and three columns of 0 and 1 for atypical angina, non-anginal pain and no pain. Keep the rows in the order of the table and use all 303 patients. Get a prediction for every patient from a five-fold stratified cross-validation, shuffled with the seed 2026, of a logistic regression with standardised columns and at most 1,000 iterations; do it once with sex and once without the column of sex. For women and for men, compare the mean predicted risk with the observed share of disease, and compute the sensitivity and the specificity at a probability of 0.5.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each model and sex, and keep the mean predicted risk of the women without sex, with three decimals, under the name "answer".  
> Test: Every prediction must come from a fold that did not hold the patient.  

In [ ]:
#@title Data of step 3.4: run this cell (it prepares the table "data" and shows it)
data = _cad_table(order="number")
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.4
teach.check_step("Step 3.4", "answer", globals())

In [ ]:
#@title Ready code of step 3.4 (run it only if the code of the assistant does not pass the check)
data = _cad_table(order="number")
d = data
pain = d["chest pain type"]
X = pd.DataFrame({"age": d["age"], "man": (d["sex"] == "man").astype(int),
                    "resting blood pressure": d["resting blood pressure"], "cholesterol": d["cholesterol"],
                    "maximum heart rate": d["maximum heart rate"], "exercise angina": d["exercise angina"],
                    "ST depression": d["ST depression"],
                    "atypical angina": (pain == "atypical angina").astype(int),
                    "non-anginal pain": (pain == "non-anginal pain").astype(int),
                    "no pain": (pain == "no pain").astype(int)})
y = d["disease"].to_numpy()
tr_ = (d["set"] == "training").to_numpy()
te_ = ~tr_
players = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
woman = (d["sex"] == "woman").to_numpy()
rows = []
for name, XX in (("with sex", X), ("without sex", X.drop(columns="man"))):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), XX, y, cv=folds, method="predict_proba")[:, 1]
    for sx, m in (("women", woman), ("men", ~woman)):
        rows.append({"model": name, "sex": sx, "mean predicted": round(p[m].mean(), 3),
                             "observed": round(y[m].mean(), 3),
                             "sensitivity": round(((p[m] >= 0.5) & (y[m] == 1)).sum() / (y[m] == 1).sum(), 3),
                             "specificity": round(((p[m] < 0.5) & (y[m] == 0)).sum() / (y[m] == 0).sum(), 3)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
answer = float(table.iloc[2]["mean predicted"])
teach.check_step("Step 3.4", "answer", globals())

**What the result shows.** Without sex the model predicts 0.417 for the women, well above the share of disease among them, and it underestimates the men. Leaving the variable out moves the errors between the sexes instead of removing them.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#ai-pair-programming-part-9-reviewing-an-explanation) and continue reading at «Working with an AI assistant, part 9: Reviewing an explanation». The weekly project can also be done on the prompt path: Its prompts are in the section «Weekly project on the prompt path» below.

## Weekly project on the prompt path

*Project of the week: [README of Week 9, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-09/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_SETS_E = {"training": "training", "validation": "validation", "testing": "test"}


def _echo():
    """The views of CAMUS of the course with their frames and expert outlines (loaded once, a few seconds)."""
    if "_echo_cache" not in globals():
        frames_ = camus.frames(camus.patients(), size=96)
        info = camus.table()
        low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
        index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
        views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 patients x 2 views
        echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
        labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
        outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # cavity or myocardium at ED or ES
        echo_y = np.array([low_ef[p] for p, v in views])
        echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
        etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"
        globals()["_echo_cache"] = dict(views=views, echo_x=echo_x, outline=outline, echo_y=echo_y, echo_set=echo_set,
                                        quality=camus.table().set_index("patient"))
    return _echo_cache


def _echo_table():
    """The 600 views of the 300 patients of CAMUS of the course, with their frames, the expert outline and an ejection fraction below 45 percent."""
    e = _echo()
    q = e["quality"]
    return pd.DataFrame({"patient": [p for p, v in e["views"]], "view": [v for p, v in e["views"]],
                         "set": [_SETS_E[s] for s in e["echo_set"]],
                         "image quality": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in e["views"]],
                         "end-diastole": list(e["echo_x"][:, 0]), "end-systole": list(e["echo_x"][:, 1]),
                         "outline": list(e["outline"].astype(np.uint8)), "low ejection fraction": e["echo_y"]})



def _project_maps():
    """The 100 test views with the probability and the Grad-CAM map of the echo network of the course (computed once, under a minute)."""
    if "_maps_cache" in globals():
        return _maps_cache
    e = _echo()
    class EchoNet(nn.Module):
        """A small convolutional network for one view: the end-diastolic and the end-systolic frame as two channels, four blocks and one output, the logit of an ejection fraction below 45 percent."""
        def __init__(self):
            super().__init__()
            sizes = [2, 16, 32, 64, 64]
            self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(sizes[i], sizes[i + 1], 3, padding=1), nn.BatchNorm2d(sizes[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                          for i in range(4)])
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

        def forward(self, x):
            return self.head(self.blocks(x)).squeeze(1)


    def train_echo(model, X, y, Xv, yv, epochs=30, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001 in groups of 32 views, each shifted by up to 4 pixels; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 32):
                b = order[i:i + 32]
                dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
                loss = loss_fn(model(torch.roll(X[b], shifts=(dy, dx), dims=(2, 3))), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, echo_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose and (epoch + 1) % 5 == 0:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def echo_predict(model, X, batch=64):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    def grad_cam(model, X, layer):
        """Grad-CAM maps of the output for a group of views: the feature maps of a layer weighted by the mean of their gradients, summed, kept above 0 and enlarged to the size of the input."""
        model.eval()
        saved = {}
        hook = layer.register_forward_hook(lambda module, inputs, output: saved.update(maps=output))
        output = model(torch.as_tensor(X))
        hook.remove()
        gradients = torch.autograd.grad(output.sum(), saved["maps"])[0]
        alpha = gradients.mean(dim=(2, 3), keepdim=True)                       # one weight per channel
        cam = torch.relu((alpha * saved["maps"]).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tuple(X.shape[-2:]), mode="bilinear", align_corners=False)[:, 0]
        return cam.detach().numpy()


    def share_inside(cam, mask):
        """The share of the weight of each map that falls inside the outline."""
        return (cam * mask).sum(axis=(1, 2)) / np.maximum(cam.sum(axis=(1, 2)), 1e-12)


    def peak_inside(cam, mask):
        """1 when the pixel with the highest value of the map lies inside the outline."""
        return np.array([m.flat[c.argmax()] for c, m in zip(cam, mask)]).astype(int)
    etr, eva, ete = [e["echo_set"] == s for s in ("training", "validation", "testing")]
    torch.manual_seed(RANDOM_STATE)
    net = EchoNet()
    train_echo(net, e["echo_x"][etr], e["echo_y"][etr], e["echo_x"][eva], e["echo_y"][eva], verbose=False)
    X = e["echo_x"][ete]
    t = _echo_table()[ete].reset_index(drop=True)
    t["probability"] = echo_predict(net, X).round(4)
    t["map"] = list(grad_cam(net, X, net.blocks[3][2]))
    globals()["_maps_cache"] = t
    return t
print("The project is ready.")

### Project step 1: The accuracy of the echo network

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee reviews a network that flags a low ejection fraction on echocardiograms and wants its accuracy for single views and for patients, with intervals, before it looks at the maps.  
> Data: The notebook holds a table named "data" with 600 views of CAMUS, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 300 patients. Each row is a view: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients) and "image quality" the grade of the experts ("Good", "Medium" or "Poor"). Each cell of "end-diastole" and of "end-systole" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of "outline" the expert outline of the left ventricular cavity and the myocardium, at end-diastole or end-systole, at the same size: 1 inside and 0 outside. "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent and 0 otherwise.  
> Task: Build a small convolutional network for one view that takes the end-diastolic and the end-systolic frame as two channels of 96 × 96 pixels. It has four blocks, and each block is a convolution with 3 × 3 kernels that keeps the size (16, 32, 64 and 64 channels), batch normalisation, ReLU and a 2 × 2 max pooling. After the blocks, the network takes the mean of every channel over the image and applies a dropout of 0.3, and a linear layer with one output gives the logit of a low ejection fraction. Train it on the training views with Adam (learning rate 0.001, weight decay 0.0001) and the binary cross-entropy for 30 epochs in shuffled groups of 32 views, shifting each group at random by up to 4 pixels in each direction, and keep the weights of the epoch with the best AUC on the validation views. Seed the random generator of the network library with 2026 before building the network and again before training. Compute the AUC on the test views, and on the test patients with the mean of the probabilities of the two views of each patient. Give each AUC a 95 percent interval from 2,000 bootstrap samples of the test views or of the test patients, from the 2.5th to the 97.5th percentile. Fix the random seed to 2026 so that the result can be repeated.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with the two AUCs and their intervals, and keep the test AUC per view, with three decimals, under the name "answer".  
> Test: No test view may be used to choose the epoch, and the bootstrap of the patients must draw patients, not views.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
data = _echo_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: The maps against the expert outline

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know whether the network looks at the left ventricle when it flags a low ejection fraction, and whether this differs between views with a low and a normal ejection fraction, between right and wrong predictions and between grades of image quality.  
> Data: The notebook holds a table named "data" with the 100 test views of CAMUS of the course, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 50 patients. Each row is a view: "patient", "view", "set" and "image quality" (the grade of the experts: "Good", "Medium" or "Poor") describe it. "end-diastole" and "end-systole" hold the frames as images of 96 × 96 pixels with values from 0 to 1, "outline" holds the expert outline of the left ventricular cavity and the myocardium at the same size, 1 inside and 0 outside, and "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent. Two columns come from the echo network of the course, trained on the training views: "probability", its probability of a low ejection fraction, and "map", its Grad-CAM map at the last block, an image of 96 × 96 values of 0 or more.  
> Task: For every view, compute three measures: the share of the sum of "map" that falls inside "outline"; 1 when the pixel with the highest value of "map" lies inside "outline", and 0 otherwise; and the share of the image that "outline" covers. A prediction is right when "probability" is at least 0.5 for a view with a low ejection fraction, or below 0.5 for a view without. Show the means of the three measures for all views, for views with a low and with a normal ejection fraction, for right and for wrong predictions and for each grade of image quality, with the number of views in each group.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the table of the groups, and keep the mean share inside the outline over all views, with three decimals, under the name "answer".  
> Test: Every share must lie between 0 and 1, and the groups of each comparison must add up to 100 views.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
data = _project_maps()   # under a minute the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: Sanity checks of the maps

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to know whether the maps depend on what the network learned. A map that stays the same when the weights of the network are replaced by random ones describes the image, not the model, and one random draw can mislead.  
> Data: The notebook holds a table named "data" with 600 views of CAMUS, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 300 patients. Each row is a view: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients) and "image quality" the grade of the experts ("Good", "Medium" or "Poor"). Each cell of "end-diastole" and of "end-systole" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of "outline" the expert outline of the left ventricular cavity and the myocardium, at end-diastole or end-systole, at the same size: 1 inside and 0 outside. "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent and 0 otherwise.  
> Task: Build a small convolutional network for one view that takes the end-diastolic and the end-systolic frame as two channels of 96 × 96 pixels. It has four blocks, and each block is a convolution with 3 × 3 kernels that keeps the size (16, 32, 64 and 64 channels), batch normalisation, ReLU and a 2 × 2 max pooling. After the blocks, the network takes the mean of every channel over the image and applies a dropout of 0.3, and a linear layer with one output gives the logit of a low ejection fraction. Train it on the training views with Adam (learning rate 0.001, weight decay 0.0001) and the binary cross-entropy for 30 epochs in shuffled groups of 32 views, shifting each group at random by up to 4 pixels in each direction, and keep the weights of the epoch with the best AUC on the validation views. Seed the random generator of the network library with 2026 before building the network and again before training. For every test view, compute the Grad-CAM map of the output at the ReLU of the last block, before its pooling. To do so, take the gradient of the output with respect to the 64 feature maps of 12 × 12 at that layer, use the mean gradient of each channel as its weight and form the weighted sum of the 64 maps. Keep only the positive part of the sum and enlarge it to 96 × 96 by bilinear interpolation. For each view, compute the share of the sum of the map that falls inside the outline. Then repeat the following for each seed from 1 to 20, twice: once for a copy of the trained network whose last block and output layer get new random weights, and once for a copy whose layers all get new random weights. Seed the random generator of the network library with the seed just before the new weights are drawn, and keep the trained network unchanged. For the copy, compute the same maps; for each view, compute the Spearman rank correlation between the 9,216 pixels of the map of the trained network and those of the map of the copy, leaving a view out when one of the two maps has the same value at every pixel, and the share of the map of the copy inside the outline. For each copy, record the mean rank correlation, the mean share inside the outline and the number of empty maps. For each kind of copy, report the median and the range over the 20 seeds of the mean rank correlation and of the mean share inside the outline, and the number of seeds in which more than half of the maps are empty, next to the mean share inside the outline of the trained network.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the results in a small table, and keep the median of the mean rank correlation for the copies with a random last block, with three decimals, under the name "answer".  
> Test: Every rank correlation must lie between −1 and 1, and the trained network must give the same maps before and after the copies are made.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
data = _echo_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Six views for the committee

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The committee wants to see the maps of a few views next to the expert outlines, including views where the network was wrong.  
> Data: The notebook holds a table named "data" with the 100 test views of CAMUS of the course, a public data set of echocardiograms: the apical two-chamber and four-chamber views of 50 patients. Each row is a view: "patient", "view", "set" and "image quality" (the grade of the experts: "Good", "Medium" or "Poor") describe it. "end-diastole" and "end-systole" hold the frames as images of 96 × 96 pixels with values from 0 to 1, "outline" holds the expert outline of the left ventricular cavity and the myocardium at the same size, 1 inside and 0 outside, and "low ejection fraction" is 1 when the ejection fraction of the patient is below 45 percent. Two columns come from the echo network of the course, trained on the training views: "probability", its probability of a low ejection fraction, and "map", its Grad-CAM map at the last block, an image of 96 × 96 values of 0 or more.  
> Task: Choose six views: two views with a low ejection fraction and a "probability" of at least 0.5, one view with a low ejection fraction and a "probability" below 0.5, one view without a low ejection fraction and a "probability" of at least 0.5, one view without a low ejection fraction and a "probability" below 0.5, and the view of image quality "Poor" with the highest "probability". Draw them in two rows of three panels: the end-diastolic frame in grey, "map" on top in a warm colour scale with partial transparency, and "outline" as a contour line. Put the patient, the view, the probability, the true class and the share of the map inside the outline in the title of each panel, and write "CAMUS, CC BY-NC-SA 4.0" under the figure.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the figure, and keep the patients and views of the six panels as a list under the name "answer".  
> Test: At least two of the six panels must show a wrong prediction.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
data = _project_maps()   # under a minute the first time
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 9: Explaining a model in code

This week a prediction is shared among the variables of a patient, and every group of variables needs its own prediction. This step shows how `itertools` lists every group of variables and every order in which they can join, how a dictionary with tuples as its keys stores the value of each group, how `math.factorial` gives the weights of the Shapley formula, how a function is handed to another function as a value, and how a chart of contributions shows them in two colours. The numbers come from the forest of Section 1 for patient 172.

In [ ]:
#@title Preparation of the Python step: run this cell once (if Section 0 has not run in this session, it runs it)
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
print("The Python step is ready.")

In [ ]:
# Python practice: a small example for learning the language
players = ["chest pain type", "exercise angina", "sex"]
for mask in itertools.product([0, 1], repeat=3):
    members = [name for name, m in zip(players, mask) if m == 1]
    print(mask, sum(mask), members)
print(len(list(itertools.product([0, 1], repeat=8))), "coalitions of eight variables")

**Python note.** `itertools.product([0, 1], repeat=3)` gives every way to write 0 or 1 three times: the eight *tuples* from `(0, 0, 0)` to `(1, 1, 1)`. A tuple is a list in round brackets that cannot be changed. Each tuple is a coalition: 1 means that the variable keeps the value of the patient. `zip(players, mask)` pairs every name with its 0 or 1, and the *list comprehension* `[name for ... if m == 1]` keeps the names of the members. `sum(mask)` counts them. With eight variables, `list(...)` holds $2^8 = 256$ tuples.

In [ ]:
# Python practice: a dictionary with tuples as keys, and the orders of the variables
value = {(0, 0, 0): 0.552, (1, 0, 0): 0.660, (0, 1, 0): 0.617, (0, 0, 1): 0.456,
         (1, 1, 0): 0.799, (1, 0, 1): 0.584, (0, 1, 1): 0.533, (1, 1, 1): 0.727}
print("value with the chest pain type alone:", value[(1, 0, 0)])
orders = list(itertools.permutations(range(3)))
print(len(orders), "orders:", orders)
contribution = {j: [] for j in range(3)}
for order in orders:
    have = [0, 0, 0]
    for j in order:
        before = value[tuple(have)]
        have[j] = 1
        contribution[j].append(round(value[tuple(have)] - before, 3))
for j in range(3):
    print(f"{players[j]:20}", contribution[j], "mean", round(np.mean(contribution[j]), 3))

**Python note.** A dictionary can use a tuple as its key, because a tuple cannot change: `value[(1, 0, 0)]` is the value of the coalition in which only the chest pain type keeps the value of the patient. `itertools.permutations(range(3))` lists the six orders of the numbers 0, 1 and 2. For each order, the list `have` starts empty, gets a 1 for each variable that joins, and `tuple(have)` turns it back into a key. The difference between the values after and before a variable joins is its contribution in that order, and the mean over the six orders is its Shapley value. The values are those of the worked example of the lecture.

In [ ]:
# Python practice: the weights of the formula, and a function handed to another function
def weight(size, n):
    return math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)


print("weights for three variables:", [round(weight(s, 3), 3) for s in range(3)])
print("sum over the 128 coalitions without one variable:",
      round(sum(weight(sum(m), 8) for m in itertools.product([0, 1], repeat=7)), 6))


def apply_to_all(function, values):
    return [function(v) for v in values]


print(apply_to_all(math.factorial, [0, 1, 2, 3, 8]))
print(apply_to_all(lambda s: round(weight(s, 8), 5), range(8)))

**Python note.** `math.factorial(n)` is $n! = 1 \times 2 \times \ldots \times n$, with $0! = 1$. `weight(size, n)` is the weight $|S|!\,(n - |S| - 1)!/n!$ of the Shapley formula. Over the 128 coalitions without a given variable, the weights add up to 1, so a Shapley value is a weighted mean. `apply_to_all` takes a *function* as its first argument and calls it for every value: A function is a value like a number and can be handed to another function. `lambda s: ...` writes a small function without a name in one line. The code of Section 2 hands the prediction of a model to the function that computes the values of the coalitions in the same way.

In [ ]:
# Python practice: a chart of contributions, red when they raise the risk and blue when they lower it
phi = {"chest pain type": 0.152, "exercise angina": 0.105, "sex": -0.083}
names = sorted(phi, key=lambda k: abs(phi[k]))
fig, ax = plt.subplots(figsize=(5.6, 1.9))
ax.barh(names, [phi[k] for k in names], color=["#9B1B1B" if phi[k] > 0 else "#1F5F8B" for k in names])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Shapley value (probability)")
plt.show()

**Python note.** `sorted(phi, key=...)` sorts the keys of the dictionary by a rule: here by the absolute value of their contributions, so that the largest bar comes last, at the top of the chart. The *conditional expression* `"#9B1B1B" if phi[k] > 0 else "#1F5F8B"` chooses a colour for each bar inside the list comprehension. `ax.axvline(0)` draws the line of no contribution.

**Quick check.** How many of the 256 coalitions of eight variables contain a given variable, such as the chest pain type?

<details><summary>Answer</summary>

128, half of them: For each of the 128 coalitions of the other seven variables, the given variable is either in or out.

</details>

**Exercise 9.1.** How many of the 256 coalitions of the eight variables have exactly three members? Keep the number.

In [ ]:
answer = None   # your answer
teach.check("Exercise 9.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(1 for m in itertools.product([0, 1], repeat=8) if sum(m) == 3)
teach.check("Exercise 9.1", answer)

**Exercise 9.2.** With the dictionary `value` of the three variables, compute the Shapley value of exercise angina, variable 1, as the mean of its six contributions. Keep it with three decimals.

In [ ]:
data = exercise_table("9.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 9.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.mean(contribution[1])), 3)
teach.check("Exercise 9.2", answer)

**Exercise 9.3.** What is the weight of a coalition with three members in the Shapley formula for eight variables? Keep it with five decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 9.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(weight(3, 8), 5)
teach.check("Exercise 9.3", answer)

## 1. The model for coronary artery disease of Week 2

*Lecture: [The black box problem: Why do clinicians not trust it?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#why-explain-interpretable-models-and-explanations-after-the-fact)*

The mini project returns to the Cleveland data of Week 2: 303 patients referred for coronary angiography, eight clinical variables and the disease on the angiogram. The same split as in Week 2 keeps a quarter of the patients for the test, and the same two models learn on the others: a logistic regression and a random forest. The case of the week is patient 172 of the test set.

**From the formulas to the code.** The model $f$: `forest` and `logistic`. The background patients $B$: `X_train`. The patient $x$: `X_test.loc[172]`. The eight players: the keys of `VARIABLES`, where the chest pain type has three columns.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(1, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X_cad, y_cad = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X_cad, y_cad, test_size=0.25, stratify=y_cad, random_state=RANDOM_STATE)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
# the eight clinical variables; the three columns of the chest pain type count as one variable
VARIABLES = {"age": ["age"], "sex": ["sex"], "resting pressure": ["trestbps"],
             "cholesterol": ["chol"], "maximum heart rate": ["thalach"],
             "exercise angina": ["exang"], "ST depression": ["oldpeak"],
             "chest pain type": ["cp_2.0", "cp_3.0", "cp_4.0"]}

print("training", X_train.shape, "| test", X_test.shape, "| share with disease in the test set", round(y_test.mean(), 3))
print("test AUC: forest", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3),
      "| logistic regression", round(roc_auc_score(y_test, logistic.predict_proba(X_test)[:, 1]), 3))

**Python note.** This is the code of Week 2 with two additions. `cad` replaces the table `data` of Week 2, so that the name stays free for the tables of the prompts. `VARIABLES` is a dictionary from the clinical name of each variable to the list of its columns: Seven variables have one column, and the chest pain type has the three dummy columns `cp_2.0`, `cp_3.0` and `cp_4.0`, with typical angina as the reference. The explanations of this week share a prediction among these eight variables, never among the three columns of one variable.

In [ ]:
CASE = 172
print(heart.loc[CASE, FEATURES + ["disease"]].to_string())
print("forest:", round(forest.predict_proba(X_test.loc[[CASE]])[0, 1], 3),
      "| logistic regression:", round(logistic.predict_proba(X_test.loc[[CASE]])[0, 1], 3),
      "| share with disease among the training patients:", round(y_train.mean(), 3))

**Python note.** `heart.loc[CASE, FEATURES + ["disease"]]` takes the row with the label 172 and the listed columns. `X_test.loc[[CASE]]`, with two pairs of brackets, keeps a table of one row, which `predict_proba` needs; with one pair it would give a single row as a series. Patient 172 is a woman (`sex` 0) without chest pain at rest (`cp` 4) whose exercise test brought on angina (`exang` 1) without ST depression (`oldpeak` 0).

**Observe and reflect.** The forest gives patient 172 a much higher probability than the logistic regression, although the two models rank the test patients almost equally well. Before reading any explanation, which of her values would you expect each model to weigh differently?

**Exercise A.** How many of the 76 test patients does the forest give a probability of coronary artery disease of at least 0.5? Keep the number.

In [ ]:
data = exercise_table("A")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((forest.predict_proba(X_test)[:, 1] >= 0.5).sum())
teach.check("Exercise A", answer)

## 2. Shapley values of one patient

*Lecture: [SHAP values: Which feature contributed how much?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#shapley-values-sharing-a-prediction-among-the-variables)*

The values of all 256 coalitions of the eight variables are computed for patient 172, with the 227 training patients as the background, and the Shapley formula turns them into eight values. For the logistic regression, the same enumeration on the scale of the log-odds is compared with the closed form.

**From the formulas to the code.** $v(S)$ for every coalition $S$: `coalition_values(predict, patient, background)`, a dictionary from a tuple of 0 and 1 to the value. $\phi_j$: `shapley(values)`. The model $f$ on the probability scale: `risk_forest` and `risk_logistic`; on the scale of the log-odds: `logodds_logistic`. The closed form $\beta_j (x_j - \bar{x}_j)$: `beta * (x - X_train.mean())`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(2, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def coalition_values(predict, patient, background, variables=VARIABLES):
    """The value of every coalition S: the mean prediction when the variables in S keep the values of the patient and the other variables take the values of each background patient. Returns a dictionary from a tuple of 0 and 1, one entry per variable and 1 for a member of S, to the value."""
    columns = list(background.columns)
    B = background.to_numpy(dtype=float)
    x = patient[columns].to_numpy(dtype=float)
    position = {name: [columns.index(c) for c in cols] for name, cols in variables.items()}
    masks = list(itertools.product([0, 1], repeat=len(variables)))
    rows = []
    for mask in masks:
        keep = np.zeros(len(columns), dtype=bool)
        for member, name in zip(mask, variables):
            if member:
                keep[position[name]] = True
        rows.append(np.where(keep, x, B))
    prediction = predict(pd.DataFrame(np.vstack(rows), columns=columns))
    means = prediction.reshape(len(masks), len(B)).mean(axis=1)
    return dict(zip(masks, means))


def shapley(values):
    """Exact Shapley values from the values of all coalitions: for each variable, the weighted mean of what it adds to the coalitions without it."""
    n = len(next(iter(values)))
    phi = np.zeros(n)
    for mask, value in values.items():
        size = sum(mask)
        for j in range(n):
            if mask[j] == 0:
                with_j = mask[:j] + (1,) + mask[j + 1:]
                weight = math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)
                phi[j] += weight * (values[with_j] - value)
    return phi


def risk_forest(d):
    return forest.predict_proba(d)[:, 1]


def risk_logistic(d):
    return logistic.predict_proba(d)[:, 1]


def logodds_logistic(d):
    return logistic.decision_function(d)

patient = X_test.loc[CASE]
values_forest = coalition_values(risk_forest, patient, X_train)
phi_forest = shapley(values_forest)
empty, full = (0,) * 8, (1,) * 8
print("starting value v(∅):", round(values_forest[empty], 4), "| prediction v(N):", round(values_forest[full], 4))
print(pd.Series(phi_forest, index=list(VARIABLES)).sort_values(key=abs, ascending=False).round(4))
print("sum of the values:", round(phi_forest.sum(), 6), "| v(N) − v(∅):", round(values_forest[full] - values_forest[empty], 6))

**Python note.** `coalition_values` builds, for each of the 256 coalitions, a table in which the columns of the members keep the values of the patient and the other columns those of the 227 training patients: `np.where(keep, x, B)` takes from `x` where `keep` is true and from `B` elsewhere, row by row. `np.vstack` stacks the 256 tables into one of 58,112 rows, so that the model predicts only once. `reshape(256, 227)` puts the predictions back into one row per coalition, and `mean(axis=1)` gives the value of each coalition. `dict(zip(masks, means))` joins the tuples and the values into a dictionary. In `shapley`, `mask[:j] + (1,) + mask[j + 1:]` builds the same coalition with variable $j$ added; `(1,)` is a tuple of one element.

In [ ]:
scaler, lr = logistic[0], logistic[1]
beta = lr.coef_[0] / scaler.scale_                       # log-odds per unit of each column
terms = beta * (patient - X_train.mean())
closed = np.array([terms[cols].sum() for cols in VARIABLES.values()])
phi_logodds = shapley(coalition_values(logodds_logistic, patient, X_train))
phi_logistic = shapley(coalition_values(risk_logistic, patient, X_train))
print(pd.DataFrame({"closed form": closed, "enumeration, log-odds": phi_logodds,
                    "enumeration, probability": phi_logistic, "forest, probability": phi_forest},
                   index=list(VARIABLES)).round(4).to_string())
print("largest difference between the closed form and the enumeration:", float(np.abs(closed - phi_logodds).max()))
order = np.argsort(np.abs(phi_forest))
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, phi, title in ((axes[0], phi_forest, "random forest"), (axes[1], phi_logistic, "logistic regression")):
    ax.barh(np.array(list(VARIABLES))[order], phi[order], color=np.where(phi[order] > 0, "#9B1B1B", "#1F5F8B"))
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Shapley value (probability)")
plt.tight_layout()
plt.show()

**Python note.** A pipeline made with `make_pipeline` can be indexed like a list: `logistic[0]` is the scaler and `logistic[1]` the logistic regression. The coefficients of the regression belong to standardised columns, and dividing them by `scaler.scale_`, the standard deviations of the training patients, turns them into log-odds per unit of each original column. `patient - X_train.mean()` subtracts the means column by column, because both are labelled with the same column names. `terms[cols].sum()` adds the terms of the columns of one variable. `np.argsort` gives the positions that sort the absolute values, so both charts list the variables in the same order.

**Observe and reflect.** In which two variables do the forest and the logistic regression disagree most for patient 172? Which of the two models would you trust for a woman of 59 years without chest pain at rest, and what would you check before deciding?

**Exercise B.** With the closed form of the logistic regression, compute the Shapley value of the resting blood pressure of patient 172 on the scale of the log-odds. Keep it with three decimals.

In [ ]:
data = exercise_table("B")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(beta[2] * (patient["trestbps"] - X_train["trestbps"].mean())), 3)
teach.check("Exercise B", answer)

## 3. From one patient to all patients

*Lecture: [From one patient to all patients](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#from-one-patient-to-all-patients)*

The Shapley values of the forest are computed for all 76 test patients, about half a minute, and summarised: the mean of the absolute values of each variable, the variable that leads for each patient, and a plot of every value against the value of its variable. The permutation importance of Week 2, computed for the eight variables, gives a second ranking.

**From the formulas to the code.** One row of Shapley values per test patient: `S_forest`, an array of 76 × 8. The mean absolute value $\frac{1}{n}\sum_i |\phi_{ij}|$: `np.abs(S_forest).mean(axis=0)`. The permutation importance of a variable: the AUC minus the AUC after shuffling its columns together.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(3, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
S_forest = np.array([shapley(coalition_values(risk_forest, X_test.iloc[i], X_train)) for i in range(len(X_test))])
S_logistic = np.array([shapley(coalition_values(risk_logistic, X_test.iloc[i], X_train)) for i in range(len(X_test))])
summary = pd.DataFrame({"forest": np.abs(S_forest).mean(axis=0), "logistic regression": np.abs(S_logistic).mean(axis=0),
                        "largest in patients (forest)": np.bincount(np.abs(S_forest).argmax(axis=1), minlength=8)},
                       index=list(VARIABLES))
print(summary.sort_values("forest", ascending=False).round(3).to_string())
rng = np.random.default_rng(RANDOM_STATE)
auc = roc_auc_score(y_test, risk_forest(X_test))
drop = {}
for name, cols in VARIABLES.items():
    losses = []
    for _ in range(20):
        shuffled = X_test.copy()
        shuffled[cols] = X_test[cols].to_numpy()[rng.permutation(len(X_test))]
        losses.append(auc - roc_auc_score(y_test, risk_forest(shuffled)))
    drop[name] = np.mean(losses)
print("permutation importance (drop in AUC):", {k: round(float(v), 3) for k, v in drop.items()})
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for k, name in enumerate(summary.sort_values("forest").index):
    j = list(VARIABLES).index(name)
    col = X_test[VARIABLES[name]].to_numpy() @ np.arange(1, len(VARIABLES[name]) + 1)
    level = (col - col.min()) / max(col.max() - col.min(), 1e-9)
    jitter = np.random.default_rng(j).uniform(-0.25, 0.25, len(col))
    ax.scatter(S_forest[:, j], k + jitter, c=level, cmap="coolwarm", s=14)
ax.set_yticks(range(8))
ax.set_yticklabels(summary.sort_values("forest").index)
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Shapley value of the forest (probability); colour: value of the variable, blue low and red high")
plt.tight_layout()
plt.show()

**Python note.** The list comprehension calls `coalition_values` and `shapley` for each test patient; `X_test.iloc[i]` takes the patient by position. `np.bincount(..., minlength=8)` counts how often each of the eight positions occurs, here how often each variable has the largest absolute value. In the permutation importance, `rng.permutation(len(X_test))` shuffles the order of the patients, and the three columns of the chest pain type are shuffled together, as one variable. In the plot, each point is one patient: `k + jitter` spreads the points of one variable vertically, and `cmap="coolwarm"` colours them from blue to red by the value of the variable; for the chest pain type, the colour runs from typical angina to no pain.

In [ ]:
# Optional: the same values for the logistic regression with the shap package, when it is installed
try:
    import shap
    background = shap.maskers.Independent(scaler.transform(X_train), max_samples=len(X_train))
    values_shap = shap.LinearExplainer(lr, background).shap_values(scaler.transform(X_test.loc[[CASE]]))[0]
    grouped = [values_shap[[list(X_train.columns).index(c) for c in cols]].sum() for cols in VARIABLES.values()]
    print("shap", shap.__version__, np.round(grouped, 4))
    print("closed form", np.round(closed, 4))
except ImportError:
    print("The shap package is not installed; the closed form above gives the same values.")
except Exception as error:
    print("The shap package could not be used here:", error)

**Advanced (optional).** The shap package of Lundberg and Lee computes Shapley values for many kinds of models. For a linear model it uses the closed form; `max_samples=len(X_train)` keeps all training patients as the background, because by default the package draws 100 of them, and its values would then differ slightly from those of the enumeration. The package treats the three columns of the chest pain type as three variables, so the code adds them up afterwards; for a linear model on the log-odds this gives exactly the value of the variable. Install the package in Anaconda with `pip install shap` if you want to try it on the forest as well.

**Observe and reflect.** The chest pain type leads the forest in most test patients, and "no pain" raises the predicted risk. What does this say about the patients of the Cleveland data, and would you expect the same pattern in a chest pain unit?

**Exercise C.** In how many of the 76 test patients is the chest pain type the variable with the largest absolute Shapley value in the forest? Keep the number.

In [ ]:
data = exercise_table("C")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((np.abs(S_forest).argmax(axis=1) == 7).sum())
teach.check("Exercise C", answer)

**Mini project: which clinical factor moves a model of the registry most?** *Lecture: [Mini project: SHAP in a CAD prediction model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#mini-project-shap-in-a-cad-prediction-model-which-clinical-factor-ldl-age-ef-influences-the-model-most)*

The Cleveland data hold no LDL and no ejection fraction. The registry of the virtual clinic holds both, for 3,600 patients, together with age, sex, smoking, diabetes, hypertension, family history and HDL. A random forest learns coronary artery disease from these nine factors on three quarters of the patients, and the exact Shapley values of 100 test patients, against a background of 100 training patients, show which factor moves its predictions most. The registry also knows the truth: the effects with which the virtual clinic drew the disease.

**From the formulas to the code.** The nine factors: `FACTORS`. The value of a coalition and the Shapley values: `coalition_values(..., variables=FACTORS)` and `shapley`, as above. The ranking: `np.abs(S_reg).mean(axis=0)`.

In [ ]:
from vclinic import make_cohort
from vclinic.cohort import TRUE_EFFECTS

registry = make_cohort(3600)
registry["male"] = (registry["sex"] == "M").astype(int)
FACTORS = {"age": ["age"], "sex": ["male"], "smoking": ["smoker"], "diabetes": ["diabetes"],
           "hypertension": ["hypertension"], "family history": ["family_history"],
           "LDL": ["ldl"], "HDL": ["hdl"], "EF": ["lvef"]}
X_reg = registry[[c for cols in FACTORS.values() for c in cols]]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, registry["cad"], test_size=0.25, stratify=registry["cad"], random_state=RANDOM_STATE)
forest_reg = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=RANDOM_STATE).fit(Xr_train, yr_train)


def risk_forest_reg(d):
    return forest_reg.predict_proba(d)[:, 1]

print("test AUC of the forest:", round(roc_auc_score(yr_test, risk_forest_reg(Xr_test)), 3))
background_reg = Xr_train.sample(100, random_state=RANDOM_STATE)
patients_reg = Xr_test.sample(100, random_state=RANDOM_STATE)
S_reg = np.array([shapley(coalition_values(risk_forest_reg, patients_reg.iloc[i], background_reg, variables=FACTORS)) for i in range(len(patients_reg))])
ranking = pd.Series(np.abs(S_reg).mean(axis=0), index=list(FACTORS)).sort_values(ascending=False)
print(ranking.round(3).to_string())
print("effects with which the clinic drew the disease (log-odds):",
      {k: v for k, v in TRUE_EFFECTS.items() if k != "intercept"})
ranking.sort_values().plot.barh(figsize=(6, 3.2), color="#9B1B1B")
plt.xlabel("mean absolute Shapley value (probability)")
plt.tight_layout()
plt.show()

**Python note.** `make_cohort(3600)` draws the 3,600 patients of the registry, and `TRUE_EFFECTS` holds the effects that the virtual clinic used to draw coronary artery disease, on the scale of the log-odds; the ejection fraction is not among them. `min_samples_leaf=10` keeps at least ten patients in every leaf of a tree, so that the probabilities are smoother. `.sample(100, random_state=RANDOM_STATE)` draws 100 rows the same way in every run. With nine factors, every patient needs 2^9 = 512 coalitions, each predicted for the 100 background patients; the cell takes about half a minute.

**Observe and reflect.** The ejection fraction moves the forest more than any risk factor, but the virtual clinic never used it to draw the disease: It lowered the ejection fraction of the patients who already had coronary artery disease. What does a Shapley value tell the committee, and what does it not tell?

**The Python path of Practice stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#lime-a-simple-model-around-one-patient) and continue reading at «LIME: Local explanations».

## 4. LIME around patient 172

*Lecture: [LIME: Local explanations](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#lime-a-simple-model-around-one-patient)*

LIME explains one prediction with a simple model that imitates the forest near the patient. Each neighbour keeps the value of patient 172 for some variables and takes the values of a random training patient for the others. A ridge regression of the risk of the forest on these choices, weighted by how close each neighbour stays to the patient, gives one coefficient per variable.

**From the formulas to the code.** The binary representation $z \in \{0, 1\}^8$: `keep`. The distance $d(z)$, the square root of the number of variables replaced: `distance`. The kernel $\pi(z) = \exp(-d(z)^2/w^2)$ with $w = 0.75\sqrt{8}$: `weight`. The coefficients of the weighted linear model: `surrogate.coef_`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(4, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def lime(predict, patient, background, variables=VARIABLES, n=2000, width=None, seed=RANDOM_STATE):
    """LIME for one patient: n neighbours in which each variable keeps the value of the patient or takes that of a random background patient, weights that fall with the number of variables replaced, and a weighted linear model whose coefficients are the explanation."""
    rng = np.random.default_rng(seed)
    columns = list(background.columns)
    B = background.to_numpy(dtype=float)
    x = patient[columns].to_numpy(dtype=float)
    width = 0.75 * np.sqrt(len(variables)) if width is None else width
    keep = rng.integers(0, 2, size=(n, len(variables)))       # 1: the variable keeps the value of the patient
    rows = B[rng.integers(0, len(B), size=n)].copy()
    for j, cols in enumerate(variables.values()):
        idx = [columns.index(c) for c in cols]
        rows[:, idx] = np.where(keep[:, [j]] == 1, x[idx], rows[:, idx])
    risk = predict(pd.DataFrame(rows, columns=columns))
    distance = np.sqrt((1 - keep).sum(axis=1))                 # square root of the number of variables replaced
    weight = np.exp(-distance ** 2 / width ** 2)
    surrogate = Ridge(alpha=1.0).fit(keep, risk, sample_weight=weight)
    return surrogate.coef_

coef = lime(risk_forest, patient, X_train)
print(pd.DataFrame({"LIME": coef, "Shapley": phi_forest}, index=list(VARIABLES)).round(3))
print("correlation between LIME and Shapley:", round(np.corrcoef(coef, phi_forest)[0, 1], 3))

**Python note.** `np.random.default_rng(seed)` makes a random generator of its own, so the neighbours do not depend on any other random draw of the notebook. `rng.integers(0, 2, size=(n, 8))` draws a table of 0 and 1, one row per neighbour, and `B[rng.integers(0, len(B), size=n)]` picks one training patient for each neighbour. Inside the loop, `keep[:, [j]]` keeps the column as a column of one, so that `np.where` can compare it with the columns of variable $j$ row by row. `Ridge(alpha=1.0)` is a linear regression with a small penalty on large coefficients, and `sample_weight=weight` makes the close neighbours count more.

In [ ]:
runs = {n: np.array([lime(risk_forest, patient, X_train, n=n, seed=s) for s in range(1, 21)]) for n in (2000, 100)}
for n, r in runs.items():
    first = pd.Series([list(VARIABLES)[j] for j in np.abs(r).argmax(axis=1)]).value_counts()
    print(f"{n} neighbours, 20 seeds: first variable", {name: int(c) for name, c in first.items()})
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, (n, r) in zip(axes, runs.items()):
    for j in range(8):
        ax.scatter(r[:, j], np.full(20, j) + np.random.default_rng(j).uniform(-0.2, 0.2, 20), color="#1F5F8B", s=10)
    ax.set_yticks(range(8))
    ax.set_yticklabels(list(VARIABLES))
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(f"{n} neighbours, 20 seeds", fontsize=10)
    ax.set_xlabel("LIME coefficient")
plt.tight_layout()
plt.show()

**Python note.** The *dictionary comprehension* `{n: ... for n in (2000, 100)}` builds a dictionary with one entry for each number of neighbours; each entry is an array of 20 rows, one per seed. `np.abs(r).argmax(axis=1)` finds, in each row, the position of the largest absolute coefficient, and `value_counts` counts how often each variable comes first; `int(c)` turns each count into a plain Python number for printing. In the chart, each variable gets one row of 20 points, one per seed, spread a little up and down so that they do not hide each other.

**Observe and reflect.** With 2,000 neighbours the twenty seeds agree, and with 100 neighbours the leading variable changes from seed to seed. If a vendor showed you a single LIME explanation of one patient, which two numbers would you ask for before reading it?

**Advanced (optional).** The kernel width $w$ decides how far the neighbourhood reaches. With a very small width only the neighbours that keep almost every value of the patient count, and the weighted model becomes flat; with a very large width every neighbour counts the same and LIME explains the forest over all patients rather than near patient 172. Try `lime(risk_forest, patient, X_train, width=1.0)` and `width=5.0` and compare the coefficients.

**Exercise D.** Compute LIME for patient 172 with a kernel width of 0.5 and the default 2,000 neighbours and seed. What is the coefficient of the chest pain type? Keep it with three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(lime(risk_forest, patient, X_train, width=0.5)[7]), 3)
teach.check("Exercise D", answer)

## 5. A network for a low ejection fraction on CAMUS

*Lecture: [Grad-CAM: Seeing where the CNN looks](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#grad-cam-where-does-a-network-look)*

Before a map can be judged, there must be a network to explain. The 300 patients of CAMUS of Week 4 give 600 views, two-chamber and four-chamber, each with an end-diastolic and an end-systolic frame of 96 × 96 pixels and the expert outline of the left ventricle. A small convolutional network learns to flag an ejection fraction below 45 percent from one view, on the training patients of CAMUS; the validation patients choose the epoch.

**From the formulas to the code.** The input $x$, two frames of one view: `echo_x[i]`, an array of 2 × 96 × 96. The output $y^c$, the logit of a low ejection fraction: `net(x)`. The expert outline $M$: `outline[i]`, true inside the cavity and the myocardium at end-diastole or end-systole.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(5, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
frames_ = camus.frames(camus.patients(), size=96)
info = camus.table()
low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 patients x 2 views
echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # cavity or myocardium at ED or ES
echo_y = np.array([low_ef[p] for p, v in views])
echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"

print(echo_x.shape, "| views by set:", {s: int((echo_set == s).sum()) for s in ("training", "validation", "testing")},
      "| share with a low ejection fraction:", round(echo_y.mean(), 3))
fig, axes = plt.subplots(1, 4, figsize=(9.6, 2.6))
for ax, i in zip(axes, [0, 1, 2, 3]):
    ax.imshow(echo_x[i, 0], cmap="gray")
    ax.contour(outline[i], levels=[0.5], colors="#E8A33D", linewidths=1)
    ax.set_title(f"{views[i][0]} {views[i][1]}, low EF {echo_y[i]}", fontsize=8)
    ax.axis("off")
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()

**Python note.** `camus.frames` reads the frames and the expert masks of the patients of Week 4 and brings them to 96 × 96 pixels. The dictionary `index` finds the position of a patient, view and phase in these arrays, and `np.stack` puts the two frames of each view into one array of 600 × 2 × 96 × 96. `np.isin(mask, (1, 2))` is true where the mask holds the cavity (1) or the myocardium (2), and `.any(axis=1)` joins end-diastole and end-systole into one outline per view. `ax.contour(..., levels=[0.5])` draws the line where the outline changes from false to true. The frames of CAMUS may be shown with the credit under the figure, and they are never shared further.

In [ ]:
class EchoNet(nn.Module):
    """A small convolutional network for one view: the end-diastolic and the end-systolic frame as two channels, four blocks and one output, the logit of an ejection fraction below 45 percent."""
    def __init__(self):
        super().__init__()
        sizes = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(sizes[i], sizes[i + 1], 3, padding=1), nn.BatchNorm2d(sizes[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


def train_echo(model, X, y, Xv, yv, epochs=30, seed=2026, verbose=True):
    """Adam with a learning rate of 0.001 in groups of 32 views, each shifted by up to 4 pixels; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = order[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            loss = loss_fn(model(torch.roll(X[b], shifts=(dy, dx), dims=(2, 3))), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, echo_predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose and (epoch + 1) % 5 == 0:
            print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def echo_predict(model, X, batch=64):
    model.eval()
    return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()

torch.manual_seed(RANDOM_STATE)
net = EchoNet()
print("weights:", sum(p.numel() for p in net.parameters()))
train_echo(net, echo_x[etr], echo_y[etr], echo_x[eva], echo_y[eva])
p_echo = echo_predict(net, echo_x)
print("test AUC per view:", round(roc_auc_score(echo_y[ete], p_echo[ete]), 3))

**Python note.** `nn.Conv2d(a, b, 3, padding=1)` slides a window of 3 × 3 pixels over `a` channels and gives `b` channels of the same size; `nn.MaxPool2d(2)` halves the height and the width, so four blocks turn 96 × 96 pixels into 6 × 6. `nn.AdaptiveAvgPool2d(1)` takes the mean of every channel, and `squeeze(1)` turns the output of shape (n, 1) into n numbers. In `train_echo`, `torch.roll` shifts every group of views by the same random number of pixels, so the network does not rely on the exact place of the heart; `copy.deepcopy(model.state_dict())` keeps a copy of the weights of the best epoch. `@torch.no_grad()` above `echo_predict` switches off the gradients, which prediction does not need.

**Observe and reflect.** The network reaches its AUC from two frames of one view, without measuring any volume. Which errors would you expect in a view of poor quality, and which in a patient whose ejection fraction is close to 45 percent?

**Exercise E.** Take the mean of the probabilities `p_echo` of the two views of each test patient and compute the AUC over the 50 test patients. Keep it with three decimals.

In [ ]:
data = exercise_table("E")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
patients = np.array([p for p, v in views])
per_patient = pd.DataFrame({"patient": patients[ete], "p": p_echo[ete], "y": echo_y[ete]}).groupby("patient").mean()
answer = round(float(roc_auc_score(per_patient["y"], per_patient["p"])), 3)
teach.check("Exercise E", answer)

## 6. Grad-CAM maps against the expert outline

*Lecture: [Grad-CAM: Seeing where the CNN looks](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#grad-cam-where-does-a-network-look)*

Grad-CAM weighs the 64 feature maps of the last block by the mean of their gradients, adds them up and keeps the positive part. The map of each test view is compared with the expert outline: the share of its weight inside the outline, the place of its strongest pixel, and the same measures for views with a low and a normal ejection fraction, for right and wrong predictions and for the three grades of image quality.

**From the formulas to the code.** The feature maps $A^k$ of the last block: `saved["maps"]`. The weights $\alpha_k = \frac{1}{Z}\sum_{i,j} \partial y^c / \partial A^k_{ij}$: `alpha`. The map $L = \mathrm{ReLU}(\sum_k \alpha_k A^k)$: `cam`. The share inside the outline $\sum M L / \sum L$: `share_inside(cam, mask)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(6, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def grad_cam(model, X, layer):
    """Grad-CAM maps of the output for a group of views: the feature maps of a layer weighted by the mean of their gradients, summed, kept above 0 and enlarged to the size of the input."""
    model.eval()
    saved = {}
    hook = layer.register_forward_hook(lambda module, inputs, output: saved.update(maps=output))
    output = model(torch.as_tensor(X))
    hook.remove()
    gradients = torch.autograd.grad(output.sum(), saved["maps"])[0]
    alpha = gradients.mean(dim=(2, 3), keepdim=True)                       # one weight per channel
    cam = torch.relu((alpha * saved["maps"]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=tuple(X.shape[-2:]), mode="bilinear", align_corners=False)[:, 0]
    return cam.detach().numpy()


def share_inside(cam, mask):
    """The share of the weight of each map that falls inside the outline."""
    return (cam * mask).sum(axis=(1, 2)) / np.maximum(cam.sum(axis=(1, 2)), 1e-12)


def peak_inside(cam, mask):
    """1 when the pixel with the highest value of the map lies inside the outline."""
    return np.array([m.flat[c.argmax()] for c, m in zip(cam, mask)]).astype(int)

cam = grad_cam(net, echo_x[ete], net.blocks[3][2])
mask = outline[ete]
inside, peak = share_inside(cam, mask), peak_inside(cam, mask)
area = mask.reshape(len(mask), -1).mean(axis=1)
quality_table = camus.table().set_index("patient")
quality = np.array([quality_table.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in views])[ete]
right = (p_echo[ete] >= 0.5) == (echo_y[ete] == 1)
groups = {"all": np.ones(len(cam), bool), "low EF": echo_y[ete] == 1, "normal EF": echo_y[ete] == 0,
          "right": right, "wrong": ~right, "Good": quality == "Good", "Medium": quality == "Medium", "Poor": quality == "Poor"}
print(pd.DataFrame({"views": [int(m.sum()) for m in groups.values()],
                    "inside the outline": [inside[m].mean() for m in groups.values()],
                    "strongest pixel inside": [peak[m].mean() for m in groups.values()],
                    "area of the outline": [area[m].mean() for m in groups.values()]}, index=list(groups)).round(3).to_string())

**Python note.** `register_forward_hook` asks PyTorch to call a small function each time the chosen layer runs; here the `lambda` stores the output of the layer, the feature maps, in the dictionary `saved`. `torch.autograd.grad(output.sum(), saved["maps"])` gives the gradient of the outputs with respect to these maps, all views at once, because the output of one view does not depend on the other views. `mean(dim=(2, 3), keepdim=True)` averages over the height and the width and keeps the shape needed to multiply each channel. `F.interpolate(..., mode="bilinear")` enlarges the 12 × 12 map to 96 × 96, and `detach()` separates the result from the gradients before it becomes a NumPy array.

In [ ]:
order = np.argsort(p_echo[ete])
chosen = [order[-1], order[-2], int(np.where(~right & (echo_y[ete] == 1))[0][0]), int(np.where(~right & (echo_y[ete] == 0))[0][0]), order[0],
          int(np.where(quality == "Poor")[0][np.argmax(p_echo[ete][quality == "Poor"])])]
test_views = [views[i] for i in np.where(ete)[0]]
fig, axes = plt.subplots(2, 3, figsize=(9, 7.6))
for ax, k in zip(axes.ravel(), chosen):
    ax.imshow(echo_x[ete][k, 0], cmap="gray")
    ax.imshow(cam[k], cmap="inferno", alpha=0.45)
    ax.contour(mask[k], levels=[0.5], colors="#7FD3F7", linewidths=1)
    ax.set_title(f"{test_views[k][0]} {test_views[k][1]}: p {p_echo[ete][k]:.2f}, low EF {echo_y[ete][k]}\ninside {inside[k]:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()

**Python note.** `np.argsort` sorts the test views by their probability, so `order[-1]` and `order[-2]` are the two most confident flags of a low ejection fraction and `order[0]` is the most confident normal view. `np.where(~right & (echo_y[ete] == 1))[0][0]` finds the first missed low ejection fraction; `~` turns true into false and back. Each panel draws the frame in grey, the map on top with `alpha=0.45`, a transparency of 45 percent, and the outline as a light blue line.

**Observe and reflect.** Look at the missed low ejection fraction and at the wrong flag of a normal view. Does a map on the heart make you trust a prediction more? What would a map have to show before you used it to explain a result to a patient or a colleague?

**Advanced (optional).** The last block gives maps of 12 × 12 cells, each 8 pixels wide. The third block, `net.blocks[2][2]`, gives maps of 24 × 24, sharper but further from the decision. Compute `grad_cam(net, echo_x[ete], net.blocks[2][2])` and compare its share inside the outline with that of the last block.

**Exercise F.** In what share of the 100 test views does the map put a larger share of its weight inside the outline than the share of the image that the outline covers? Keep the share with two decimals.

In [ ]:
data = exercise_table("F")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float((inside > area).mean()), 2)
teach.check("Exercise F", answer)

## 7. Sanity checks with randomised networks

*Lecture: [Sanity checks: Is the map about the model?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#sanity-checks-is-the-map-about-the-model)*

The model randomisation test of Adebayo and colleagues gives a copy of the trained network new random weights, first in the last block and the head, then in every layer, and computes the maps again. Maps that hardly change describe the image, not what the network learned. One random draw can mislead, so the test is repeated with the seeds 1 to 20.

**From the formulas to the code.** The trained map $L$ and the map $\tilde{L}$ of a randomised copy: `cam` and `cam_copy`. The Spearman rank correlation $\rho(L, \tilde{L})$ over the 9,216 pixels of a view: `spearmanr(a.ravel(), b.ravel())`. One row of the table `draws` for each seed and kind of copy.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(7, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def randomised(model, seed, part):
    """A copy of the network with new random weights in the last block and the head (part "top") or in every layer (part "all")."""
    copy_ = copy.deepcopy(model)
    torch.manual_seed(seed)
    layers = list(copy_.blocks[3].modules()) + list(copy_.head.modules()) if part == "top" else [m for m in copy_.modules() if m is not copy_]
    for layer in layers:
        if hasattr(layer, "reset_parameters"):
            layer.reset_parameters()
    return copy_


def rank_correlation(A, B):
    return np.array([spearmanr(a.ravel(), b.ravel()).correlation if a.std() > 0 and b.std() > 0 else np.nan for a, b in zip(A, B)])


rows = []
for part in ("top", "all"):
    for seed in range(1, 21):
        copy_net = randomised(net, seed, part)
        cam_copy = grad_cam(copy_net, echo_x[ete], copy_net.blocks[3][2])
        rho = rank_correlation(cam, cam_copy)
        rows.append({"part": part, "seed": seed, "rho": np.nanmean(rho) if np.isfinite(rho).any() else np.nan,
                     "inside": share_inside(cam_copy, mask).mean(), "empty": int((cam_copy.reshape(len(cam_copy), -1).sum(axis=1) == 0).sum())})
draws = pd.DataFrame(rows)
print(draws.groupby("part")[["rho", "inside"]].agg(["median", "min", "max"]).round(3).to_string())
print("trained network: share inside", round(inside.mean(), 3), "| area of the outline", round(area.mean(), 3))

**Python note.** `copy.deepcopy(model)` makes a full copy of the network, so the trained network `net` keeps its weights. `modules()` lists the network and all the layers inside it; `reset_parameters()` gives a layer new random weights, as when it was built, and `hasattr` checks that the layer has this method, because ReLU and pooling layers have no weights. `a.std() > 0` leaves out a map whose pixels all have the same value, for which a rank correlation is not defined; `np.nan` marks it as missing, and `np.nanmean` skips it. `groupby("part")[...].agg(["median", "min", "max"])` summarises the 20 draws of each kind of copy.

In [ ]:
k = chosen[0]
top_1, all_1 = randomised(net, 1, "top"), randomised(net, 1, "all")
one = echo_x[ete][k:k + 1]
fig, axes = plt.subplots(1, 3, figsize=(8.4, 3))
for ax, (title, m) in zip(axes, [("trained", cam[k]),
                                  ("random last block, seed 1", grad_cam(top_1, one, top_1.blocks[3][2])[0]),
                                  ("all weights random, seed 1", grad_cam(all_1, one, all_1.blocks[3][2])[0])]):
    ax.imshow(echo_x[ete][k, 0], cmap="gray")
    ax.imshow(m, cmap="inferno", alpha=0.45)
    ax.contour(mask[k], levels=[0.5], colors="#7FD3F7", linewidths=1)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()
fig, ax = plt.subplots(figsize=(6.4, 2.6))
for y_, part in enumerate(("top", "all")):
    values = draws.loc[draws["part"] == part, "inside"]
    ax.scatter(values, np.full(len(values), y_) + np.random.default_rng(y_).uniform(-0.12, 0.12, len(values)), color="#1F5F8B", s=16)
ax.axvline(inside.mean(), color="#9B1B1B", label="trained network")
ax.axvline(area.mean(), color="grey", ls="--", label="area of the outline")
ax.set_yticks([0, 1])
ax.set_yticklabels(["random last block", "all weights random"])
ax.set_xlabel("share of the map inside the outline, 20 seeds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The first figure draws one view three times: the map of the trained network, of a copy with a random last block and of a copy with all weights random, each made with seed 1. `echo_x[ete][k:k + 1]` keeps a group of one view, because `grad_cam` expects a group, and each copy is made once, so that the layer handed to `grad_cam` belongs to the same copy. The second figure draws one point for each of the 20 copies of each kind, with the share of the trained network as a red line and the area of the outline as a dashed line.

**Observe and reflect.** Several random copies put as much of their maps inside the outline as the trained network. What, then, does a high share inside the outline prove, and what would you ask a vendor to show in addition?

**Exercise G.** Among the 20 copies with all weights random, in how many are more than half of the 100 maps empty? Use the column `empty` of `draws`.

In [ ]:
data = exercise_table("G")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(((draws["part"] == "all") & (draws["empty"] > 50)).sum())
teach.check("Exercise G", answer)

**The Python path of Practice stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#last-two-hours-ai-reliability-and-clinical-validation) and continue reading at «Last two hours: AI reliability and clinical validation».

## 8. Dataset shift between two hospitals

*Lecture: [Distribution shift (data drift): Pitfalls](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#dataset-shift-what-changed-and-does-it-matter)*

The risk model of the chest pain unit of Week 8 learns at hospital A from the clinical table, the troponin, the echocardiogram and the triage note, and is then used at hospital B. Before any outcome of B is known, a classifier that tries to tell the presentations of B from those of A shows whether the inputs changed, and which inputs changed. A classifier that cannot tell the hospitals apart reaches an AUC near 0.5.

**From the formulas to the code.** The inputs $x$ of every presentation: `inputs(A)` and `inputs(B)`, 39 columns. The label of the hospital $h$: `hospital`, 0 for A and 1 for B. The AUC of the domain classifier, from five-fold cross-validation: `domain_auc(columns)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(8, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
    """Standardise the columns and fit a logistic regression; with a validation set, keep the inverse regularisation strength with the largest validation AUC."""
    best = None
    for C in (Cs if Xv is not None else (1.0,)):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
        auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
        if best is None or auc > best[1]:
            best = (model, auc)
    return best[0]


def logit(p):
    """The log-odds of a probability."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def bootstrap_auc(y, p, n=2000, seed=2026):
    """AUC in n bootstrap samples of the patients: an array of n values."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.array(out)


def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
    """The AUC of p1 minus the AUC of p2 in n bootstrap samples of the same patients: an array of n differences."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return np.array(out)


def calibration(y, p):
    """Calibration in the large (the intercept with the slope fixed at 1) and the calibration slope."""
    z = logit(p)
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
    return a, slope


def reliability(y, p, bins=10):
    """Mean predicted risk and observed share of events in groups of equal size: two arrays."""
    order = np.argsort(p)
    groups = np.array_split(order, bins)
    return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])


def share_threshold(p, k):
    """The threshold that sends home the k patients with the lowest predicted risk: the k-th lowest risk."""
    return float(np.sort(p)[k - 1])


def rule_out_threshold(y, p, sensitivity):
    """The largest threshold that keeps the sensitivity for the events at least at the given level. Patients at or below it are discharged. It is chosen on the validation set."""
    order = np.argsort(p)
    missed = np.cumsum(y[order])              # events at or below each threshold
    ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
    return float(p[order][ok.max()]) if len(ok) else 0.0


def wilson(k, n, z=1.96):
    """The 95 percent Wilson interval of a proportion k/n."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)


def net_benefit(y, p, thresholds):
    """Net benefit of treating the patients over each threshold: true positives minus false positives weighted by the odds of the threshold, per patient."""
    n = len(y)
    return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])

chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
A = chest[chest["hospital"] == "A"].reset_index(drop=True)
B = chest[chest["hospital"] == "B"].reset_index(drop=True)
tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
            "heart_rate", "sbp", "egfr"]


def clinical(d):
    """The clinical table: age, sex, risk factors, known coronary artery disease, heart rate, systolic pressure and eGFR."""
    return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


def laboratory(d):
    """Troponin in multiples of the upper reference limit of its assay: the log of the value at arrival and after one hour, the change in one hour, and whether the second value exists."""
    url = d["troponin_url"]
    t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
    return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                         "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


def echo(d):
    """The bedside echocardiogram: whether it was done, the ejection fraction (60 when not done) and a regional wall motion abnormality (0 when not done)."""
    return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


def note(d):
    """The triage note as 20 numbers: TF-IDF of words and word pairs, reduced by a singular value decomposition fitted on the training notes."""
    return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

inputs = lambda d: pd.concat([clinical(d), laboratory(d), echo(d), note(d)], axis=1)
XA, XB = inputs(A), inputs(B)
unit_model = fit_logistic(XA[tr], y[tr], Xv=XA[va], yv=y[va])
pA, pB = unit_model.predict_proba(XA)[:, 1], unit_model.predict_proba(XB)[:, 1]

for name, yy, pp in (("hospital A, test", y[te], pA[te]), ("hospital B", yB, pB)):
    intercept, slope = calibration(yy, pp)
    print(f"{name:22} AUC {roc_auc_score(yy, pp):.3f}   mean predicted {pp.mean():.3f}   observed {yy.mean():.3f}   intercept {intercept:.2f}   slope {slope:.2f}")

**Python note.** This cell repeats the tools of Week 8 and the model of the chest pain unit without the ECG: `clinical`, `laboratory`, `echo` and `note` turn a table of presentations into numbers, and `inputs` joins them into 39 columns. `fit_logistic` chooses the strength of the penalty on the validation presentations of A. `calibration` returns the intercept, the shift of the logits that makes the mean predicted risk equal to the share of events, and the slope. The loop runs over three *tuples* at once: In `for name, yy, pp in (...)`, each tuple is unpacked into three names.

In [ ]:
hospital_X = pd.concat([XA, XB], ignore_index=True)
hospital = np.r_[np.zeros(len(XA)), np.ones(len(XB))]          # 0: hospital A, 1: hospital B
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


def domain_auc(columns):
    """How well a logistic regression on the given columns tells hospital B from hospital A, with predictions from five-fold cross-validation."""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    p = cross_val_predict(model, hospital_X[columns], hospital, cv=folds, method="predict_proba")[:, 1]
    return roc_auc_score(hospital, p)

groups = {"all inputs": list(hospital_X.columns), "clinical table": CLINICAL,
          "troponin": ["log_trop_0h", "log_trop_1h", "delta_1h", "has_1h"], "echocardiogram": ["echo_done", "lvef", "rwma"],
          "triage note": [f"note_{i + 1}" for i in range(20)]}
for name, columns in groups.items():
    print(f"{name:26} AUC of the domain classifier {domain_auc(columns):.3f}")
words = lambda d: d["note_tr"].str.split().str.len()
print(pd.DataFrame({"A": [words(A).mean(), A["age"].mean(), A["diabetes"].mean(), A["egfr"].mean()],
                    "B": [words(B).mean(), B["age"].mean(), B["diabetes"].mean(), B["egfr"].mean()]},
                   index=["words in the note", "age", "diabetes", "eGFR"]).round(2))

**Python note.** `pd.concat([XA, XB], ignore_index=True)` stacks the inputs of the two hospitals, and `np.r_[np.zeros(len(XA)), np.ones(len(XB))]` writes the label of the hospital for each row. `cross_val_predict` gives every presentation a probability from a model that did not see it, so the AUC is not inflated. The lambda `words` splits each note at the spaces with `.str.split()` and counts the pieces with `.str.len()`.

**Observe and reflect.** The note alone tells the two hospitals apart almost perfectly, while the clinical table does so only in part. Which of these differences would worry you for a risk model, the shorter notes or the older patients with more diabetes and a lower eGFR, and why?

**Exercise H.** How well does the domain classifier tell the hospitals apart without the 20 columns of the note? Use `domain_auc` with the other 19 columns and keep the AUC with three decimals.

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(domain_auc([c for c in hospital_X.columns if not c.startswith("note_")])), 3)
teach.check("Exercise H", answer)

## 9. Observed against expected events at hospital B

*Lecture: [Monitoring after deployment: Observed against expected events](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#monitoring-after-deployment-observed-against-expected-events)*

After deployment, the outcomes of the patients arrive one after the other. The cumulative difference between the observed and the expected events stays near 0 for a calibrated model, within a band that widens with the number of patients. When the curve leaves the band for good, the model expects too few or too many events, and the committee updates its intercept with the first local patients.

**From the formulas to the code.** The difference $D_m = \sum_{i \le m} (y_i - p_i)$: `difference`. The band $1.96\sqrt{\sum_{i \le m} p_i(1 - p_i)}$: `band`. The update of the intercept, a shift $a$ of the logits: `calibration(yB[:600], pB[:600])[0]`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(9, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def observed_minus_expected(y_seq, p_seq):
    """The cumulative observed minus expected events over the presentations in their order, and the band of 1.96 standard deviations that a calibrated model would stay within."""
    difference = np.cumsum(y_seq) - np.cumsum(p_seq)
    band = 1.96 * np.sqrt(np.cumsum(p_seq * (1 - p_seq)))
    return difference, band

difference, band = observed_minus_expected(yB, pB)
first = int(np.where(difference > band)[0][0]) + 1
print("first presentation above the band:", first, "| after 1,000 presentations:",
      round(difference[-1], 1), "±", round(band[-1], 1))
shift_600, _ = calibration(yB[:600], pB[:600])
p_600 = 1 / (1 + np.exp(-(logit(pB) + shift_600)))
difference_600, band_600 = observed_minus_expected(yB[600:], p_600[600:])
print("update with the first 600 presentations: shift of the logits", round(shift_600, 3),
      "| presentations 601 to 1,000:", round(difference_600[-1], 1), "±", round(band_600[-1], 1))
order = np.arange(1, len(yB) + 1)
fig, ax = plt.subplots(figsize=(7.6, 3.2))
ax.fill_between(order, -band, band, color="#1F5F8B", alpha=0.15, label="band of a calibrated model")
ax.plot(order, difference, color="#9B1B1B", label="model of A")
ax.plot(order[600:], difference_600, color="#0E7A78", label="updated with the first 600")
ax.axvline(first, color="grey", ls="--", lw=0.8)
ax.set_xlabel("order of arrival at hospital B")
ax.set_ylabel("observed − expected events")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** `np.cumsum` adds the values one after the other, so `np.cumsum(y_seq) - np.cumsum(p_seq)` is the difference after each presentation. `np.where(difference > band)[0][0]` is the position of the first presentation above the band; adding 1 turns the position, which starts at 0, into the order of arrival, which starts at 1. The update keeps the slope of the model and adds the same number to every logit: `logit(pB) + shift_600` and the function $1/(1 + e^{-z})$ give the new probabilities. The updated curve starts again at presentation 601, so it is not judged on the patients that set the update.

**Observe and reflect.** The curve of B leaves its band after about a quarter of the presentations and never comes back. Who should see this chart at a hospital, how often, and what should happen on the day of the first alarm?

**Exercise I.** Update the model with the first 300 presentations of B instead of 600, and follow the updated curve from presentation 301. At which presentation does the difference first lie outside the band, above `band` or below `-band`? Keep its order of arrival.

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
shift_300, _ = calibration(yB[:300], pB[:300])
p_300 = 1 / (1 + np.exp(-(logit(pB) + shift_300)))
difference_300, band_300 = observed_minus_expected(yB[300:], p_300[300:])
answer = int(np.where(np.abs(difference_300) > band_300)[0][0]) + 301
teach.check("Exercise I", answer)

**Calibration and a confidence score.** *Lecture: [Model calibration and confidence score](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#model-calibration-and-confidence-score)*

The forest of the mini project gives every test patient of the registry a probability of coronary artery disease. Its calibration is judged as in Week 8, and a *confidence score*, how far the probability lies from the threshold of 0.5, says how sure the model is about each decision. The table shows what happens when the model decides only for the patients on whom it is confident and refers the others to a cardiologist.

**From the formulas to the code.** The confidence of a probability $p$: $\max(p, 1 - p)$ in `coverage_table`. The share of patients decided by the model at a level $c$: the mean of `confidence >= c`.

In [ ]:
def coverage_table(y, p, levels=(0.5, 0.6, 0.7, 0.8, 0.9)):
    """For every level of confidence: the share of patients whose confidence max(p, 1 - p) reaches it, and the accuracy of the decision at 0.5 among them."""
    confidence = np.maximum(p, 1 - p)
    rows = []
    for level in levels:
        keep = confidence >= level
        rows.append({"confidence at least": level, "share of patients": round(float(keep.mean()), 3),
                     "accuracy": round(float(((p[keep] >= 0.5) == y[keep]).mean()), 3) if keep.any() else np.nan})
    return pd.DataFrame(rows)

p_reg = np.clip(risk_forest_reg(Xr_test), 0.001, 0.999)
y_reg = yr_test.to_numpy()
intercept, slope = calibration(y_reg, p_reg)
print("AUC", round(roc_auc_score(y_reg, p_reg), 3), "· Brier", round(float(np.mean((p_reg - y_reg) ** 2)), 3),
      "· intercept", round(intercept, 2), "· slope", round(slope, 2))
print(coverage_table(y_reg, p_reg).to_string(index=False))

**Python note.** `np.clip` keeps the probabilities of the forest between 0.001 and 0.999, because a leaf in which all patients had the disease gives 1, and the logit of 1 is infinite. `coverage_table` computes the confidence of every patient once and then, for each level, keeps the patients whose confidence reaches it: `keep` is an array of True and False. `(p[keep] >= 0.5) == y[keep]` marks the right decisions among them, and its mean is their accuracy.

**Observe and reflect.** At which level of confidence would you let the model decide alone, and who would see the patients below it? What would change if the model were not calibrated?

## 10. Subgroups and fairness through unawareness

*Lecture: [Bias detection: Sex, age, ethnicity](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#fairness-does-the-model-work-for-every-patient)*

A threshold that sends home the patients at the lowest risk is chosen on the validation presentations of A so that it keeps a sensitivity of 98 percent. The test presentations then show whether it keeps this promise for women, for men, for patients under 65 and for patients aged 65 or more. The last cell returns to the Cleveland data and removes sex from the logistic regression, an approach called fairness through unawareness.

**From the formulas to the code.** The threshold $t$: `rule_out_threshold(y[va], pA[va], 0.98)`. The sensitivity in a group $g$, $1 - \text{missed}_g / \text{events}_g$: the column of the table. The predictions without sex, from five-fold cross-validation: `cross_val_predict(...)` on `X_cad.drop(columns="sex")`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB09_reliability_and_explainable_ai.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(10, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
threshold = rule_out_threshold(y[va], pA[va], 0.98)
subgroups = {"all": np.ones(len(A), bool), "women": (A["sex"] == "F").to_numpy(), "men": (A["sex"] == "M").to_numpy(),
             "under 65": (A["age"] < 65).to_numpy(), "65 or older": (A["age"] >= 65).to_numpy()}
rows = []
for name, m in subgroups.items():
    m = m & te
    home = pA[m] <= threshold
    rows.append({"group": name, "n": int(m.sum()), "MACE": int(y[m].sum()), "mean predicted": pA[m].mean(),
                 "observed": y[m].mean(), "sent home": int(home.sum()), "MACE among them": int(y[m][home].sum()),
                 "sensitivity": 1 - y[m][home].sum() / y[m].sum()})
print("threshold", round(threshold, 4))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Python note.** `rule_out_threshold` sorts the validation presentations by risk and takes the largest risk at which the events at or below it stay within 2 percent of all events. In the loop, `m = m & te` keeps the test presentations of each group, and `pA[m] <= threshold` marks those sent home. `to_string(index=False)` prints the table without the numbers of its rows.

In [ ]:
folds_cad = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, X_ in (("with sex", X_cad), ("without sex", X_cad.drop(columns="sex"))):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_, y_cad, cv=folds_cad, method="predict_proba")[:, 1]
    for sex_name, code_ in (("women", 0), ("men", 1)):
        m = (X_cad["sex"] == code_).to_numpy()
        yy, pp = y_cad.to_numpy()[m], p[m]
        rows.append({"model": name, "sex": sex_name, "mean predicted": pp.mean(), "observed": yy.mean(),
                     "sensitivity": ((pp >= 0.5) & (yy == 1)).sum() / (yy == 1).sum(),
                     "specificity": ((pp < 0.5) & (yy == 0)).sum() / (yy == 0).sum(),
                     "false positives": int(((pp >= 0.5) & (yy == 0)).sum()), "false negatives": int(((pp < 0.5) & (yy == 1)).sum())})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Python note.** The loop fits the same logistic regression twice with five-fold cross-validation, once with all ten columns and once with `X_cad.drop(columns="sex")`. `(pp >= 0.5) & (yy == 1)` marks the true positives; adding them with `.sum()` and dividing by the patients with disease gives the sensitivity. The inner loop splits the 303 patients into women and men by the column `sex`, which stays available for the comparison even when the model does not use it.

**Observe and reflect.** Without sex, the model predicts a much higher risk for the women of the Cleveland data than the share of disease among them, and it misses more disease in men. Would you remove a variable from a model to make it fair? What would you report instead, and to whom?

**Exercise J.** What is the AUC of the risk model `pA` in the test presentations of women? Keep it with three decimals.

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
women = (A["sex"] == "F").to_numpy() & te
answer = round(float(roc_auc_score(y[women], pA[women])), 3)
teach.check("Exercise J", answer)

**The Python path of Practice stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lecture.html#ai-pair-programming-part-9-reviewing-an-explanation) and continue reading at «Working with an AI assistant, part 9: Reviewing an explanation». The rest of this Jupyter notebook holds further practice and the help for the weekly project.

## Application challenges by field

| Field | Challenge |
|---|---|
| Emergency cardiology | Explain the probability of MACE for the test presentations of A that were sent home at the threshold of Section 10 and had a MACE. Use the closed form of the logistic regression on the 39 inputs and group them into the clinical table, the troponin, the echocardiogram and the note. Which group pushed each of these patients below the threshold? |
| Interventional cardiology | Compare the Shapley values of the forest for the test patients with typical angina and with no pain. Which variables decide the prediction in each group, and how does the selection of patients for angiography in the Cleveland data shape the answer? |
| Heart failure | Train the echo network on the four-chamber views only and compare its AUC and its Grad-CAM maps with those of the network trained on both views. Do the maps of the four-chamber network fall more often on the left ventricle? |
| Clinical research | Run the second test of Adebayo and colleagues: Train the echo network again on labels shuffled at random among the training views, and compare its maps with those of the trained network by the rank correlation and the share inside the outline. What does the network learn from labels without meaning? |
| Women and the heart | In the Cleveland data, compare the Shapley values of sex in the forest for women and men among the test patients. Then look at the share of disease by chest pain type in women and men. How much of what the model learns from sex reflects who was referred for angiography? |
| Quality and safety | Write the monitoring plan for the risk model of the chest pain unit at hospital B in half a page: what is charted, how often and by whom, the alarm, the first steps after an alarm, the rule for an update, and when the model is withdrawn. |
| Medical informatics | Write a model card for the echo network of this week: intended use, data and their licence, performance with intervals and by image quality, the method of explanation and its sanity check over seeds, known limits, the plan for monitoring, and a contact. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-09/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of this week extends Sections 5 to 7: The echo network is judged with intervals, its Grad-CAM maps are compared with the expert outlines in groups of views, the sanity check is repeated over 20 seeds, and six views go to the committee with half a page on what the maps can and cannot support.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>